<a href="https://colab.research.google.com/github/sebadelza98-design/Balance-de-masa/blob/main/Balance_Masa_Agua_Colab_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 💧 Balance de Masa Hídrico — Coca-Cola Chile### Versión Google Colab (Streamlit vía túnel)## ⚠️ IMPORTANTE: usa **Entorno de ejecución → Ejecutar todo**Eso corre todas las celdas en orden automáticamente. Si las corres a mano, hazlo **de arriba hacia abajo, sin saltarte ninguna**.Al final aparecerá una **URL `https://....trycloudflare.com`** → ábrela y sube tu Excel dentro de la app.> Los 7 archivos `.py` están embebidos. Solo subes el Excel desde la app.

## 1. Instalar dependencias

In [1]:
!pip install -q streamlit pandas numpy networkx plotly openpyxl
print('✅ Dependencias instaladas')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 116.0 MB/s eta 0:00:00
✅ Dependencias instaladas


## 2. Crear los archivos del proyectoCada celda escribe un archivo en `/content`. Ejecútalas todas.

In [2]:
%%writefile models.py
"""
models.py
Modelos de datos: Stream, UnitOperation, PlantModel
"""
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Dict, List, Optional
import uuid
import networkx as nx
import pandas as pd


# ─────────────────────────────────────────────
# Stream
# ─────────────────────────────────────────────

STREAM_TYPES = ["normal", "recirculacion", "rechazo", "perdida", "producto"]

STREAM_COLORS = {
    "normal":       "rgba(33,150,243,0.65)",
    "recirculacion":"rgba(255,152,0,0.65)",
    "rechazo":      "rgba(244,67,54,0.65)",
    "perdida":      "rgba(158,158,158,0.65)",
    "producto":     "rgba(76,175,80,0.65)",
}

@dataclass
class Stream:
    """Corriente de agua entre unidades de operación."""
    id: str = field(default_factory=lambda: "S" + str(uuid.uuid4())[:5].upper())
    name: str = ""
    source_id: Optional[str] = None
    target_id: Optional[str] = None
    flow: Optional[float] = None      # m³/h  — None = desconocido
    temperature: Optional[float] = None   # °C
    conductivity: Optional[float] = None  # µS/cm
    tds: Optional[float] = None           # mg/L
    stream_type: str = "normal"
    description: str = ""

    @property
    def is_unknown(self) -> bool:
        return self.flow is None

    def display_flow(self) -> str:
        return f"{self.flow:.2f}" if self.flow is not None else "?"


# ─────────────────────────────────────────────
# UnitOperation
# ─────────────────────────────────────────────

UNIT_TYPES = [
    "pozo", "filtro", "nano", "uf", "osmosis", "tanque",
    "desaireador", "ablandador", "caldera", "cip",
    "lavadora", "condensador", "torre", "mezclador", "perdida", "otro"
]

UNIT_COLORS = {
    "pozo":        "#1565C0",
    "filtro":      "#0288D1",
    "nano":        "#00838F",
    "uf":          "#00838F",
    "osmosis":     "#2E7D32",
    "tanque":      "#6A1B9A",
    "desaireador": "#4527A0",
    "ablandador":  "#5C6BC0",
    "caldera":     "#B71C1C",
    "cip":         "#E65100",
    "lavadora":    "#EF6C00",
    "condensador": "#00695C",
    "torre":       "#1B5E20",
    "mezclador":   "#37474F",
    "perdida":     "#424242",
    "otro":        "#546E7A",
}

@dataclass
class UnitOperation:
    """Unidad de operación en la planta de tratamiento."""
    id: str = field(default_factory=lambda: "U" + str(uuid.uuid4())[:5].upper())
    name: str = ""
    op_type: str = "otro"
    description: str = ""
    capacity: Optional[float] = None   # m³/h — CAUDAL nominal (bombas, filtros, membranas)
    volume_m3: Optional[float] = None  # m³  — VOLUMEN de almacenamiento (tanques/estanques)
    fill_level: Optional[float] = None # fracción de llenado operativo 0-1 (nivel ~constante)
    height_m: Optional[float] = None   # m — altura total del estanque (para fijar nivel en metros)
    recovery: Optional[float] = None   # fracción de recuperación LIMPIA (membranas)
    fouling: float = 0.0               # 0-1 ensuciamiento; baja la recuperación efectiva
    accumulation: Optional[float] = None  # m³/h — >0 llenándose, <0 vaciándose. None = régimen estacionario
    x_pos: float = 0.0
    y_pos: float = 0.0

    @property
    def operating_volume_m3(self) -> Optional[float]:
        """Volumen operativo = volumen × nivel de llenado (m³)."""
        if self.volume_m3 is None:
            return None
        nivel = self.fill_level if self.fill_level is not None else 1.0
        return self.volume_m3 * nivel

    @property
    def level_m(self) -> Optional[float]:
        """Nivel actual (m) = altura total × llenado (si se conoce la altura)."""
        if self.height_m is None or self.fill_level is None:
            return None
        return self.height_m * self.fill_level

    @property
    def effective_recovery(self) -> Optional[float]:
        """Recuperación efectiva = recuperación limpia × (1 − ensuciamiento)."""
        if self.recovery is None:
            return None
        return max(0.0, self.recovery * (1.0 - (self.fouling or 0.0)))

    def residence_time_h(self, flow_m3h: float) -> Optional[float]:
        """Tiempo de residencia hidráulico τ = V_operativo / Q (horas).

        Válido con nivel ~constante (acumulación ≈ 0). Devuelve None si
        no hay volumen definido o el caudal es nulo.
        """
        vop = self.operating_volume_m3
        if vop is None or not flow_m3h or flow_m3h <= 0:
            return None
        return vop / flow_m3h


# ─────────────────────────────────────────────
# Helpers de acumulación (term. transitorio en estanques)
# ─────────────────────────────────────────────

def accumulation_from_volume(v_inicial_m3: float, v_final_m3: float,
                             delta_t_horas: float) -> float:
    """Acumulación [m³/h] = ΔVolumen / Δtiempo."""
    if delta_t_horas <= 0:
        return 0.0
    return (v_final_m3 - v_inicial_m3) / delta_t_horas


def accumulation_from_level(nivel_inicial_m: float, nivel_final_m: float,
                            area_m2: float, delta_t_horas: float) -> float:
    """Acumulación [m³/h] = (Δnivel × área transversal) / Δtiempo."""
    if delta_t_horas <= 0:
        return 0.0
    return (nivel_final_m - nivel_inicial_m) * area_m2 / delta_t_horas


def accumulation_from_flows(caudal_entrada_m3h: float,
                            caudal_salida_m3h: float) -> float:
    """Acumulación [m³/h] = caudal entrada − caudal salida."""
    return caudal_entrada_m3h - caudal_salida_m3h


# ─────────────────────────────────────────────
# PlantModel
# ─────────────────────────────────────────────

class PlantModel:
    """Modelo de la planta: grafo dirigido de nodos + corrientes."""

    def __init__(self, name: str = "Planta de Tratamiento de Agua"):
        self.name = name
        self.units: Dict[str, UnitOperation] = {}
        self.streams: Dict[str, Stream] = {}
        self.graph = nx.DiGraph()
        self._inlets:  Dict[str, List[str]] = {}   # unit_id → [stream_ids]
        self._outlets: Dict[str, List[str]] = {}   # unit_id → [stream_ids]

    # ── Node management ──────────────────────

    def add_unit(self, unit: UnitOperation) -> None:
        self.units[unit.id] = unit
        self.graph.add_node(unit.id)
        self._inlets.setdefault(unit.id, [])
        self._outlets.setdefault(unit.id, [])

    def remove_unit(self, unit_id: str) -> None:
        if unit_id not in self.units:
            return
        for sid in list(self._inlets.get(unit_id, []) + self._outlets.get(unit_id, [])):
            self._detach_stream(sid)
        self.graph.remove_node(unit_id)
        del self.units[unit_id]
        self._inlets.pop(unit_id, None)
        self._outlets.pop(unit_id, None)

    # ── Stream management ─────────────────────

    def add_stream(self, stream: Stream) -> None:
        self.streams[stream.id] = stream
        if stream.source_id and stream.target_id:
            if stream.source_id in self.units and stream.target_id in self.units:
                self.graph.add_edge(stream.source_id, stream.target_id,
                                    stream_id=stream.id)
                self._outlets[stream.source_id].append(stream.id)
                self._inlets[stream.target_id].append(stream.id)

    def _detach_stream(self, sid: str) -> None:
        s = self.streams.get(sid)
        if not s:
            return
        if s.source_id in self._outlets:
            self._outlets[s.source_id] = [x for x in self._outlets[s.source_id] if x != sid]
        if s.target_id in self._inlets:
            self._inlets[s.target_id] = [x for x in self._inlets[s.target_id] if x != sid]

    def remove_stream(self, sid: str) -> None:
        if sid not in self.streams:
            return
        self._detach_stream(sid)
        del self.streams[sid]

    def set_flow(self, sid: str, value: Optional[float]) -> None:
        if sid in self.streams:
            self.streams[sid].flow = value

    # ── Accessors ─────────────────────────────

    def inlets_of(self, unit_id: str) -> List[Stream]:
        return [self.streams[s] for s in self._inlets.get(unit_id, [])
                if s in self.streams]

    def outlets_of(self, unit_id: str) -> List[Stream]:
        return [self.streams[s] for s in self._outlets.get(unit_id, [])
                if s in self.streams]

    def process_units(self) -> List[UnitOperation]:
        """Unidades con al menos una entrada Y una salida (escriben ecuaciones)."""
        return [
            u for uid, u in self.units.items()
            if self._inlets.get(uid) and self._outlets.get(uid)
        ]

    def source_units(self) -> List[UnitOperation]:
        return [u for uid, u in self.units.items() if not self._inlets.get(uid)]

    def sink_units(self) -> List[UnitOperation]:
        return [u for uid, u in self.units.items() if not self._outlets.get(uid)]

    # ── DataFrames ────────────────────────────

    def streams_df(self) -> pd.DataFrame:
        rows = []
        for sid, s in self.streams.items():
            src = self.units.get(s.source_id)
            tgt = self.units.get(s.target_id)
            rows.append({
                "ID": sid,
                "Nombre": s.name,
                "Desde": src.name if src else "",
                "Hacia": tgt.name if tgt else "",
                "Caudal (m³/h)": s.flow,
                "Tipo": s.stream_type,
                "Temp (°C)": s.temperature,
                "Conductividad": s.conductivity,
                "SDT (mg/L)": s.tds,
            })
        return pd.DataFrame(rows) if rows else pd.DataFrame()

    def units_df(self) -> pd.DataFrame:
        rows = [{"ID": u.id, "Nombre": u.name, "Tipo": u.op_type,
                 "Caudal nom. (m³/h)": u.capacity, "Volumen (m³)": u.volume_m3,
                 "Recuperación": u.recovery, "Descripción": u.description}
                for u in self.units.values()]
        return pd.DataFrame(rows) if rows else pd.DataFrame()

    def summary(self) -> dict:
        known = sum(1 for s in self.streams.values() if not s.is_unknown)
        unk   = sum(1 for s in self.streams.values() if s.is_unknown)
        return {
            "nodos": len(self.units),
            "corrientes": len(self.streams),
            "conocidas": known,
            "desconocidas": unk,
        }


Writing models.py


In [3]:
%%writefile solver.py
"""
solver.py
MassBalanceSolver — resuelve corrientes desconocidas mediante álgebra lineal.

Balance para cada unidad de proceso (en estado estacionario, sin acumulación):
    Σ entradas - Σ salidas = 0

Se construye la matriz de incidencia A y se resuelve A·x = b.
"""
from __future__ import annotations
from typing import Dict, List, Optional, Tuple
import numpy as np
from models import PlantModel


class MassBalanceSolver:
    """Resuelve el balance de masa de un PlantModel."""

    DEFAULT_TOLERANCE = 0.05  # m³/h — para datos de diseño exactos

    def __init__(self, model: PlantModel, tolerance: float = None):
        self.model = model
        self.tolerance = tolerance if tolerance is not None else self.DEFAULT_TOLERANCE

    @property
    def TOLERANCE(self):
        return self.tolerance

    # ─────────────────────────────────────────
    # Interfaz principal
    # ─────────────────────────────────────────

    def solve(self) -> Tuple[Dict[str, float], Dict]:
        """
        Retorna (solved_flows, diagnostics).
        solved_flows: {stream_id: valor_m3h}
        diagnostics: dict con estado, mensajes y residuos.
        """
        m = self.model
        stream_ids  = list(m.streams.keys())
        proc_units  = m.process_units()

        diag = {
            "status": "ok",
            "messages": [],
            "residuals": {},
            "n_eq": 0,
            "n_unk": 0,
        }

        if not proc_units:
            diag["status"] = "sin_ecuaciones"
            diag["messages"].append("No hay unidades de proceso (nodos con entradas y salidas).")
            return {sid: m.streams[sid].flow for sid in stream_ids}, diag

        # ── Índices ──
        n_units  = len(proc_units)
        n_streams = len(stream_ids)
        s_idx = {sid: j for j, sid in enumerate(stream_ids)}

        # ── Matriz de incidencia ──
        A = np.zeros((n_units, n_streams))
        for i, unit in enumerate(proc_units):
            for s in m.inlets_of(unit.id):
                A[i, s_idx[s.id]] = +1.0
            for s in m.outlets_of(unit.id):
                A[i, s_idx[s.id]] = -1.0

        # ── Vector de acumulación (término transitorio en estanques) ──
        # Balance: Σ entradas − Σ salidas = Acumulación  →  A·x = acc
        acc_vec = self._accumulation_vector(proc_units)
        if np.any(acc_vec != 0):
            nodos_acc = [proc_units[i].name for i in range(n_units) if acc_vec[i] != 0]
            diag["accumulation"] = {
                proc_units[i].name: round(float(acc_vec[i]), 4)
                for i in range(n_units) if acc_vec[i] != 0
            }
            diag["messages"].append(
                f"♒ Acumulación activa en: {', '.join(nodos_acc)}")

        # ── Separar conocidos / desconocidos ──
        unk_idx = [j for j, sid in enumerate(stream_ids) if m.streams[sid].is_unknown]
        kno_idx = [j for j, sid in enumerate(stream_ids) if not m.streams[sid].is_unknown]

        diag["n_unk"] = len(unk_idx)
        diag["n_eq"]  = n_units

        # Si todo conocido → verificar consistencia
        if not unk_idx:
            return self._check_consistency(A, stream_ids, proc_units, diag, acc_vec)

        # ── Construir sistema: A_unk · x_unk = b ──
        A_unk = A[:, unk_idx]
        A_kno = A[:, kno_idx]
        x_kno = np.array([m.streams[stream_ids[j]].flow for j in kno_idx], dtype=float)
        b = acc_vec - A_kno @ x_kno

        # ── Restricciones de recuperación ──
        extra_A, extra_b = self._recovery_constraints(unk_idx, kno_idx, stream_ids, x_kno)
        if extra_A.size:
            A_unk = np.vstack([A_unk, extra_A])
            b     = np.concatenate([b, extra_b])

        n_eq, n_u = A_unk.shape
        diag["n_eq"] = n_eq

        # ── Diagnóstico de especificación ──
        if n_eq < n_u:
            diag["status"] = "sub_especificado"
            faltan = self._undetermined_streams(A_unk, unk_idx, stream_ids)
            diag["faltantes"] = faltan
            _msg = (f"⚠️ Sistema sub-especificado: {n_eq} ecuaciones para {n_u} "
                    f"incógnitas. Faltan {n_u - n_eq} dato(s).")
            if faltan:
                _msg += (" Corrientes sin determinar (ingresa alguna como dato): "
                         + ", ".join(faltan) + ".")
            diag["messages"].append(_msg)
        elif n_eq > n_u:
            diag["messages"].append(
                f"ℹ️ Sistema sobre-especificado: {n_eq} ecuaciones para {n_u} incógnitas. "
                "Usando mínimos cuadrados.")

        # ── Resolver ──
        x_unk, residuals, rank, _ = np.linalg.lstsq(A_unk, b, rcond=None)

        # Verificar residuos
        res_vec = A_unk @ x_unk - b
        max_res = float(np.max(np.abs(res_vec)))
        if max_res > self.TOLERANCE:
            diag["status"] = "inconsistente"
            diag["messages"].append(
                f"❌ Balance inconsistente. Residuo máximo: {max_res:.3f} m³/h — "
                "revisar datos de entrada.")

        # Flujos negativos
        for i, val in enumerate(x_unk):
            if val < -self.TOLERANCE:
                sid  = stream_ids[unk_idx[i]]
                name = m.streams[sid].name
                diag["messages"].append(
                    f"⚠️ Corriente '{name}' resultó negativa ({val:.2f} m³/h).")

        # Verificar residuos por unidad de proceso (descontando acumulación)
        res_node = A[:, unk_idx] @ x_unk + A_kno @ x_kno - acc_vec
        for i, unit in enumerate(proc_units):
            r = float(res_node[i])
            if abs(r) > self.TOLERANCE:
                diag["residuals"][unit.name] = round(r, 4)
                if diag["status"] == "ok":
                    diag["status"] = "inconsistente"

        # ── Ensamblar resultado ──
        solved: Dict[str, float] = {}
        for j, sid in enumerate(stream_ids):
            if m.streams[sid].is_unknown:
                local_i = unk_idx.index(j)
                solved[sid] = max(0.0, float(x_unk[local_i]))
            else:
                solved[sid] = float(m.streams[sid].flow)

        if not diag["messages"] and diag["status"] == "ok":
            diag["messages"].append("✅ Balance cerrado correctamente.")

        return solved, diag

    # ─────────────────────────────────────────
    # KPIs
    # ─────────────────────────────────────────

    def compute_kpis(self, solved: Dict[str, float]) -> Dict:
        m = self.model
        kpis: Dict = {}

        # Flujos por tipo de nodo sumando corrientes de salida/entrada
        def out_total(op_type: str) -> float:
            total = 0.0
            for uid, u in m.units.items():
                if u.op_type == op_type:
                    for s in m.outlets_of(uid):
                        total += solved.get(s.id, s.flow or 0)
            return total

        def in_total(op_type: str) -> float:
            total = 0.0
            for uid, u in m.units.items():
                if u.op_type == op_type:
                    for s in m.inlets_of(uid):
                        total += solved.get(s.id, s.flow or 0)
            return total

        # Consumo de pozos = suma de corrientes de SALIDA de nodos "pozo"
        pozos = out_total("pozo")

        # Agua de producto = corrientes hacia nodos "perdida" tipo producto
        # En la práctica: suma inlets de todos los nodos tipo lavadora/cip/condensador/etc.
        useful_types = {"lavadora", "cip", "condensador", "torre", "perdida"}
        agua_util = 0.0
        for uid, u in m.units.items():
            if u.op_type in useful_types:
                agua_util += sum(solved.get(s.id, s.flow or 0) for s in m.inlets_of(uid))

        # Perdidas = corrientes tipo "rechazo" o "perdida" hacia nodos "perdida"
        perdidas_types = {"rechazo", "perdida"}
        perdidas = sum(
            solved.get(sid, s.flow or 0)
            for sid, s in m.streams.items()
            if s.stream_type in perdidas_types
        )

        # LB02: nodo cuyo ID o nombre comienza con "LB02"
        lb02 = sum(
            solved.get(s.id, s.flow or 0)
            for uid, u in m.units.items()
            if uid.upper().startswith("LB02") or u.name.upper().replace("\n", " ").startswith("LB02")
            for s in m.inlets_of(uid)
        )
        # LB03: nodo cuyo ID o nombre comienza con "LB03"
        lb03 = sum(
            solved.get(s.id, s.flow or 0)
            for uid, u in m.units.items()
            if uid.upper().startswith("LB03") or u.name.upper().replace("\n", " ").startswith("LB03")
            for s in m.inlets_of(uid)
        )
        # Benedictino: nodo tipo "otro" con ID BENED_TOTAL o similar
        bened = sum(
            solved.get(s.id, s.flow or 0)
            for uid, u in m.units.items()
            if "BENED" in uid.upper() and u.op_type in ("otro", "mezclador")
            for s in m.inlets_of(uid)
        )

        agua_util_lb = lb02 + lb03 + bened if (lb02 + lb03) > 0 else agua_util

        # Pozos principales (excluye POZO_1B y PER de la eficiencia del sistema principal)
        pozos_main = sum(
            solved.get(s.id, s.flow or 0)
            for uid, u in m.units.items()
            if u.op_type == "pozo" and uid in ("POZOS", "POTABLE")
            for s in m.outlets_of(uid)
        ) or pozos

        eficiencia = (agua_util_lb / pozos * 100) if pozos > 0 else 0.0

        # Rechazo final: flujos que entran a nodos tipo "perdida" (drains reales)
        rechazo = sum(
            solved.get(s.id, s.flow or 0)
            for uid, u in m.units.items()
            if u.op_type == "perdida"
            for s in m.inlets_of(uid)
        )

        kpis.update({
            "consumo_pozos":       round(pozos, 2),
            "agua_proceso_lb02":   round(lb02, 2),
            "agua_servicio_lb03":  round(lb03, 2),
            "agua_benedictino":    round(bened, 2),
            "agua_util_total":     round(agua_util_lb, 2),
            "rechazo_total":       round(rechazo, 2),
            "perdidas_totales":    round(pozos - agua_util_lb if pozos > 0 else 0, 2),
            "eficiencia_pct":      round(eficiencia, 1),
        })
        return kpis

    # ─────────────────────────────────────────
    # Tiempo de residencia (nivel ~constante)
    # ─────────────────────────────────────────

    def compute_residence_times(self, solved: Dict[str, float]) -> Dict[str, Dict]:
        """Tiempo de residencia hidráulico por estanque: τ = V_operativo / Q.

        Q = caudal que atraviesa el tanque (Σ entradas, estado estacionario).
        Supone nivel ~constante (acumulación ≈ 0) y usa el volumen operativo
        (volumen × nivel de llenado). No interviene en el balance.
        """
        m = self.model
        out: Dict[str, Dict] = {}
        for uid, u in m.units.items():
            if u.op_type != "tanque" or u.volume_m3 is None:
                continue
            q = sum(solved.get(s.id, s.flow or 0) for s in m.inlets_of(uid))
            tau = u.residence_time_h(q)
            out[uid] = {
                "nombre":    u.name.replace("\n", " "),
                "vol_m3":    round(u.volume_m3, 1),
                "nivel":     u.fill_level if u.fill_level is not None else 1.0,
                "vol_op_m3": round(u.operating_volume_m3, 1) if u.operating_volume_m3 is not None else None,
                "Q_m3h":     round(q, 2),
                "tau_h":     round(tau, 2) if tau is not None else None,
            }
        return out

    # ─────────────────────────────────────────
    # Eficiencia de equipos de membrana (recuperación)
    # ─────────────────────────────────────────

    MEMBRANE_TYPES = {"nano", "uf", "osmosis"}

    def compute_equipment_efficiency(self, solved: Dict[str, float]) -> Dict[str, Dict]:
        """Recuperación de los equipos de membrana (NF, RO, UF/WUR).

        recuperación = permeado / alimentación, con:
          alimentación = Σ entradas
          rechazo      = Σ salidas tipo 'rechazo'
          permeado     = Σ salidas productivas (ni 'rechazo' ni 'perdida')

        Incluye la recuperación de diseño (`recovery`) si está definida.
        """
        m = self.model
        out: Dict[str, Dict] = {}
        for uid, u in m.units.items():
            if u.op_type not in self.MEMBRANE_TYPES:
                continue
            feed = sum(solved.get(s.id, s.flow or 0) for s in m.inlets_of(uid))
            rechazo = sum(solved.get(s.id, s.flow or 0)
                          for s in m.outlets_of(uid) if s.stream_type == "rechazo")
            permeado = sum(solved.get(s.id, s.flow or 0)
                           for s in m.outlets_of(uid)
                           if s.stream_type not in ("rechazo", "perdida"))
            rec = (permeado / feed * 100) if feed > 0 else None
            out[uid] = {
                "nombre":   u.name.replace("\n", " "),
                "tipo":     u.op_type,
                "feed":     round(feed, 2),
                "permeado": round(permeado, 2),
                "rechazo":  round(rechazo, 2),
                "recuperacion_pct":        round(rec, 1) if rec is not None else None,
                "recuperacion_diseno_pct": round(u.recovery * 100, 1) if u.recovery is not None else None,
            }
        return out

    # ─────────────────────────────────────────
    # Ensuciamiento de membranas (fouling)
    # ─────────────────────────────────────────

    # Corrientes que pasan a modo recovery-driven por membrana al simular fouling
    _FOULING_FREE = {
        "NANO":   ["S13"],          # rechazo a drenaje (permeado S11 ya es incógnita)
        "WUR_01": ["S14"],          # permeado (rechazo S15 ya es incógnita)
        "WUR_02": ["S06", "S27"],   # permeado y rechazo (feed fijado aguas arriba)
        "RO_PTR": [],               # ya recovery-driven
    }

    def simulate_fouling(self, fouling_by_unit, clean_recovery=None):
        """Escenario de ensuciamiento. Fija la recuperación LIMPIA de cada
        membrana y aplica φ (recuperación efectiva = limpia·(1−φ)), pasando
        sus corrientes a modo recovery-driven. No modifica el modelo base.
        Devuelve (solved, kpis, diag)."""
        import copy
        m = copy.deepcopy(self.model)
        clean = clean_recovery or {}
        for uid, free in self._FOULING_FREE.items():
            if uid not in m.units:
                continue
            if uid in clean:
                m.units[uid].recovery = clean[uid]
            m.units[uid].fouling = float(fouling_by_unit.get(uid, 0.0) or 0.0)
            for sid in free:
                m.set_flow(sid, None)
        solver = MassBalanceSolver(m, tolerance=self.tolerance)
        solved, diag = solver.solve()
        kpis = solver.compute_kpis(solved)
        return solved, kpis, diag

    # ─────────────────────────────────────────
    # Análisis automático de pérdidas
    # ─────────────────────────────────────────

    # Tipos de unidad que RECUPERAN agua (su rechazo no es pérdida final)
    RECOVERY_TYPES = {"nano", "uf", "osmosis"}

    @staticmethod
    def _loss_category(name: str, stream_type: str) -> str:
        """Clasifica el tipo de pérdida según nombre/tipo de corriente."""
        n = name.lower()
        if "rechazo" in n and any(k in n for k in ("nano", "ro", "osmosis", "wur", "membrana")):
            return "Rechazo membrana"
        if "retro" in n or "retrolavado" in n:
            return "Retrolavado filtro"
        if "purga" in n or "torre" in n:
            return "Purga torre enfriamiento"
        if "cip" in n:
            return "Drenaje CIP"
        if "rechazo" in n or stream_type == "rechazo":
            return "Rechazo"
        return "Pérdida proceso"

    def analyze_losses(self, solved: Dict[str, float]) -> Dict:
        """
        Detecta y clasifica automáticamente las pérdidas de agua a partir
        del balance resuelto. Genera un ranking de mayor a menor.

        Una corriente es PÉRDIDA si:
          - va hacia un nodo tipo "perdida" (drenaje), o
          - es tipo "rechazo"/"perdida" y su destino NO es un equipo de
            recuperación (nano/uf/osmosis).

        Retorna:
          {
            "ranking": [ {id, corriente, destino, categoria, caudal,
                          pct_pozos, pct_perdidas}, ... ],   # ordenado desc
            "total_perdidas": float,
            "consumo_pozos": float,
            "pct_perdida_global": float,
            "por_categoria": { categoria: caudal_total, ... },
          }
        """
        m = self.model

        # Consumo de pozos (denominador)
        pozos = 0.0
        for uid, u in m.units.items():
            if u.op_type == "pozo":
                for s in m.outlets_of(uid):
                    pozos += solved.get(s.id, s.flow or 0)

        losses: List[Dict] = []
        for sid, s in m.streams.items():
            flow = solved.get(sid, s.flow or 0)
            if flow <= 1e-6:
                continue
            tgt = m.units.get(s.target_id)
            tgt_type = tgt.op_type if tgt else ""

            is_loss = False
            if tgt_type == "perdida":
                is_loss = True
            elif s.stream_type in ("rechazo", "perdida") and tgt_type not in self.RECOVERY_TYPES:
                is_loss = True

            if not is_loss:
                continue

            losses.append({
                "id":         sid,
                "corriente":  s.name,
                "destino":    tgt.name.replace("\n", " ") if tgt else "",
                "tipo":       s.stream_type,
                "categoria":  self._loss_category(s.name, s.stream_type),
                "caudal":     round(flow, 3),
            })

        total = sum(l["caudal"] for l in losses)

        # Porcentajes
        for l in losses:
            l["pct_pozos"]    = round(l["caudal"] / pozos * 100, 1) if pozos > 0 else 0.0
            l["pct_perdidas"] = round(l["caudal"] / total * 100, 1) if total > 0 else 0.0

        losses.sort(key=lambda x: -x["caudal"])

        # Agregado por categoría
        por_categoria: Dict[str, float] = {}
        for l in losses:
            por_categoria[l["categoria"]] = round(
                por_categoria.get(l["categoria"], 0.0) + l["caudal"], 3
            )

        return {
            "ranking":            losses,
            "total_perdidas":     round(total, 3),
            "consumo_pozos":      round(pozos, 3),
            "pct_perdida_global": round(total / pozos * 100, 1) if pozos > 0 else 0.0,
            "por_categoria":      por_categoria,
        }

    # ─────────────────────────────────────────
    # Helpers privados
    # ─────────────────────────────────────────

    def _check_consistency(
        self, A, stream_ids, proc_units, diag, acc_vec=None
    ) -> Tuple[Dict[str, float], Dict]:
        m = self.model
        x = np.array([m.streams[sid].flow or 0.0 for sid in stream_ids])
        if acc_vec is None:
            acc_vec = np.zeros(len(proc_units))
        res = A @ x - acc_vec
        for i, unit in enumerate(proc_units):
            r = float(res[i])
            if abs(r) > self.TOLERANCE:
                diag["residuals"][unit.name] = round(r, 4)
                diag["status"] = "inconsistente"
                diag["messages"].append(
                    f"❌ '{unit.name}': desbalance = {r:+.3f} m³/h")
        if diag["status"] == "ok":
            diag["messages"].append("✅ Todas las corrientes son conocidas y el balance cierra.")
        solved = {sid: m.streams[sid].flow or 0.0 for sid in stream_ids}
        return solved, diag

    # ─────────────────────────────────────────
    # Vector de acumulación
    # ─────────────────────────────────────────

    def _accumulation_vector(self, proc_units) -> np.ndarray:
        """
        Acumulación [m³/h] por unidad de proceso.
        Solo aplica a unidades con `accumulation` definida (típicamente tanques).
        0.0 = régimen estacionario (comportamiento por defecto, sin cambios).
        """
        return np.array([
            float(u.accumulation)
            if getattr(u, "accumulation", None) is not None
            else 0.0
            for u in proc_units
        ])

    def _undetermined_streams(self, A_unk, unk_idx, stream_ids) -> List[str]:
        """Corrientes incógnita que el sistema NO puede determinar (espacio nulo
        de la matriz). Indican qué datos faltan para cerrar el balance."""
        m = self.model
        try:
            if A_unk.size == 0:
                free = [True] * len(unk_idx)
            else:
                _u, s, vh = np.linalg.svd(A_unk, full_matrices=True)
                tol = max(A_unk.shape) * (s[0] if s.size else 0.0) * 1e-10
                rank = int((s > tol).sum())
                null_rows = vh[rank:]
                if null_rows.size == 0:
                    return []
                free = [bool(np.any(np.abs(null_rows[:, j]) > 1e-8))
                        for j in range(len(unk_idx))]
            return ["%s — %s" % (stream_ids[unk_idx[j]],
                                 m.streams[stream_ids[unk_idx[j]]].name.replace("\n", " "))
                    for j in range(len(unk_idx)) if free[j]]
        except Exception:
            return ["%s — %s" % (stream_ids[j], m.streams[stream_ids[j]].name.replace("\n", " "))
                    for j in unk_idx]

    def _recovery_constraints(
        self,
        unk_idx: List[int],
        kno_idx: List[int],
        stream_ids: List[str],
        x_kno: np.ndarray,
    ) -> Tuple[np.ndarray, np.ndarray]:
        """Ecuaciones adicionales de recuperación para membranas (RO, Nano, UF)."""
        m = self.model
        n_unk = len(unk_idx)
        rows, rhs = [], []

        for uid, unit in m.units.items():
            if unit.recovery is None:
                continue
            if unit.op_type not in ("osmosis", "nano", "uf"):
                continue
            inlets  = m.inlets_of(uid)
            outlets = m.outlets_of(uid)
            if len(inlets) != 1 or len(outlets) < 2:
                continue

            feed_id = inlets[0].id
            perm_id = outlets[0].id   # primer outlet = permeado
            r = unit.effective_recovery
            if r is None:
                continue

            row   = np.zeros(n_unk)
            rhs_v = 0.0
            s_idx = {sid: j for j, sid in enumerate(stream_ids)}

            j_perm = s_idx.get(perm_id)
            j_feed = s_idx.get(feed_id)
            if j_perm is None or j_feed is None:
                continue

            # permeado - r * feed = 0
            if j_perm in unk_idx:
                row[unk_idx.index(j_perm)] += 1.0
            elif j_perm in kno_idx:
                rhs_v -= m.streams[perm_id].flow or 0.0

            if j_feed in unk_idx:
                row[unk_idx.index(j_feed)] -= r
            elif j_feed in kno_idx:
                rhs_v += r * (m.streams[feed_id].flow or 0.0)

            if np.any(row != 0):
                rows.append(row)
                rhs.append(rhs_v)

        if rows:
            return np.array(rows), np.array(rhs)
        return np.empty((0, n_unk)), np.empty(0)


def fouling_from_time(hours_since_cip, rate_per_h, model="lineal", phi_max=0.6):
    """φ(t) en función de las horas desde el último CIP.

    - 'lineal':       φ = min(phi_max, rate_per_h · t)
    - 'exponencial':  φ = phi_max · (1 − exp(−rate_per_h · t))
    rate_per_h se calibra con datos (decaimiento de la recuperación)."""
    import math
    t = max(0.0, float(hours_since_cip))
    if model == "exponencial":
        return phi_max * (1.0 - math.exp(-rate_per_h * t))
    return min(phi_max, rate_per_h * t)


def fouling_from_cip(days_since_cip, cip_interval_days, phi_max=0.5, model="lineal"):
    """φ según el ciclo de CIP: 0 recién lavada → φ_max al cumplir el intervalo.
    El intervalo de CIP es un supuesto editable (frecuencia de lavado)."""
    import math
    if not cip_interval_days or cip_interval_days <= 0:
        return 0.0
    frac = max(0.0, float(days_since_cip)) / float(cip_interval_days)
    if model == "exponencial":
        return phi_max * (1.0 - math.exp(-3.0 * frac))
    return phi_max * min(1.0, frac)


Writing solver.py


In [4]:
%%writefile visualizer.py
"""
visualizer.py
SankeyVisualizer — diagramas Sankey y grafo de proceso con Plotly.
"""
from __future__ import annotations
from typing import Dict
import plotly.graph_objects as go
from models import PlantModel, STREAM_COLORS, UNIT_COLORS


class SankeyVisualizer:
    """Genera visualizaciones Plotly para el balance de masa."""

    # ─────────────────────────────────────────
    # Diagrama Sankey
    # ─────────────────────────────────────────

    @staticmethod
    def create_sankey(
        model: PlantModel,
        solved: Dict[str, float],
        min_flow: float = 0.05,
    ) -> go.Figure:
        """Sankey interactivo.  Nodos = equipos, links = corrientes."""

        units = list(model.units.values())
        unit_idx = {u.id: i for i, u in enumerate(units)}

        labels = [u.name for u in units]
        node_colors = [UNIT_COLORS.get(u.op_type, "#546E7A") for u in units]

        sources, targets, values, link_labels, link_colors = [], [], [], [], []

        for sid, s in model.streams.items():
            if s.source_id not in unit_idx or s.target_id not in unit_idx:
                continue
            flow = solved.get(sid, s.flow or 0.0)
            if flow < min_flow:
                continue
            sources.append(unit_idx[s.source_id])
            targets.append(unit_idx[s.target_id])
            values.append(round(flow, 3))
            link_labels.append(f"{s.name}: {flow:.2f} m³/h")
            link_colors.append(
                STREAM_COLORS.get(s.stream_type, "rgba(150,150,150,0.40)")
            )

        fig = go.Figure(
            go.Sankey(
                arrangement="snap",
                node=dict(
                    pad=14,
                    thickness=18,
                    line=dict(color="rgba(255,255,255,0.25)", width=0.5),
                    label=labels,
                    color=node_colors,
                    hovertemplate="%{label}<extra></extra>",
                ),
                link=dict(
                    source=sources,
                    target=targets,
                    value=values,
                    label=link_labels,
                    color=link_colors,
                    hovertemplate="%{label}<extra></extra>",
                ),
            )
        )

        fig.update_layout(
            title=dict(text="Balance de Masa — Diagrama Sankey", font=dict(size=15)),
            font_size=10,
            height=520,
            margin=dict(l=15, r=15, t=45, b=15),
            paper_bgcolor="rgba(15,23,42,1)",
            plot_bgcolor="rgba(15,23,42,1)",
            font_color="#E2E8F0",
        )
        return fig

    # ─────────────────────────────────────────
    # Diagrama de proceso (grafo network)
    # ─────────────────────────────────────────

    @staticmethod
    def create_process_graph(
        model: PlantModel,
        solved: Dict[str, float],
    ) -> go.Figure:
        """Grafo de proceso con nodos coloreados y etiquetas de caudal."""

        fig = go.Figure()

        # ── Aristas (corrientes) ──
        for sid, s in model.streams.items():
            src = model.units.get(s.source_id)
            tgt = model.units.get(s.target_id)
            if not (src and tgt):
                continue
            flow = solved.get(sid, s.flow or 0.0)
            color = STREAM_COLORS.get(s.stream_type, "rgba(150,150,150,0.65)")
            width = max(1.5, min(9.0, flow / 12.0))

            # Línea de la corriente
            fig.add_trace(
                go.Scatter(
                    x=[src.x_pos, None, tgt.x_pos],
                    y=[src.y_pos, None, tgt.y_pos],
                    mode="lines",
                    line=dict(color=color, width=width),
                    hoverinfo="skip",
                    showlegend=False,
                )
            )

            # Etiqueta de caudal en midpoint
            mx = (src.x_pos + tgt.x_pos) / 2
            my = (src.y_pos + tgt.y_pos) / 2
            flow_str = f"{flow:.1f}" if flow else "?"
            fig.add_annotation(
                x=mx,
                y=my,
                text=f"<b>{flow_str}</b>",
                showarrow=False,
                font=dict(size=7.5, color=color),
                bgcolor="rgba(15,23,42,0.85)",
                bordercolor=color,
                borderwidth=1,
                borderpad=2,
            )

        # ── Nodos (unidades) ──
        for uid, u in model.units.items():
            clr = UNIT_COLORS.get(u.op_type, "#546E7A")
            in_flow = sum(
                solved.get(s.id, s.flow or 0)
                for s in model.inlets_of(uid)
            )
            out_flow = sum(
                solved.get(s.id, s.flow or 0)
                for s in model.outlets_of(uid)
            )
            hover = (
                f"<b>{u.name}</b><br>"
                f"Tipo: {u.op_type}<br>"
                f"Entrada: {in_flow:.2f} m³/h<br>"
                f"Salida: {out_flow:.2f} m³/h"
                + (f"<br>Volumen: {u.volume_m3:g} m³" if getattr(u, 'volume_m3', None) else "")
                + (f"<br>Caudal nom.: {u.capacity:g} m³/h" if u.capacity else "")
            )

            fig.add_trace(
                go.Scatter(
                    x=[u.x_pos],
                    y=[u.y_pos],
                    mode="markers+text",
                    marker=dict(
                        symbol="square",
                        size=30,
                        color=clr,
                        opacity=0.90,
                        line=dict(color="rgba(255,255,255,0.35)", width=1.5),
                    ),
                    text=[u.name],
                    textposition="bottom center",
                    textfont=dict(size=7, color="#CBD5E1"),
                    hovertemplate=hover + "<extra></extra>",
                    showlegend=False,
                )
            )

        # ── Leyenda manual de tipos de corriente ──
        for stype, color in STREAM_COLORS.items():
            fig.add_trace(
                go.Scatter(
                    x=[None],
                    y=[None],
                    mode="lines",
                    name=stype.capitalize(),
                    line=dict(color=color, width=3),
                )
            )

        fig.update_layout(
            title=dict(text="Diagrama de Proceso", font=dict(size=14, color="#E2E8F0")),
            xaxis=dict(visible=False, range=[-0.03, 1.07]),
            yaxis=dict(visible=False, range=[-0.08, 1.05]),
            height=580,
            margin=dict(l=5, r=5, t=42, b=5),
            paper_bgcolor="rgba(15,23,42,1)",
            plot_bgcolor="rgba(22,33,55,1)",
            font_color="#E2E8F0",
            legend=dict(
                orientation="h",
                yanchor="top",
                y=-0.02,
                xanchor="left",
                x=0,
                font=dict(size=9),
            ),
        )
        return fig

    # ─────────────────────────────────────────
    # Gauge de eficiencia
    # ─────────────────────────────────────────

    @staticmethod
    def create_efficiency_gauge(eficiencia_pct: float) -> go.Figure:
        """Gauge circular para la eficiencia de recuperación."""
        color = (
            "#22C55E" if eficiencia_pct >= 80
            else "#EAB308" if eficiencia_pct >= 65
            else "#EF4444"
        )
        fig = go.Figure(
            go.Indicator(
                mode="gauge+number+delta",
                value=eficiencia_pct,
                delta={"reference": 75, "valueformat": ".1f"},
                number={"suffix": "%", "font": {"size": 40, "color": color}},
                title={"text": "Eficiencia de Recuperación", "font": {"size": 13, "color": "#94A3B8"}},
                gauge={
                    "axis": {"range": [0, 100], "tickcolor": "#94A3B8"},
                    "bar": {"color": color, "thickness": 0.25},
                    "bgcolor": "rgba(30,41,59,1)",
                    "borderwidth": 0,
                    "steps": [
                        {"range": [0, 65], "color": "rgba(239,68,68,0.15)"},
                        {"range": [65, 80], "color": "rgba(234,179,8,0.15)"},
                        {"range": [80, 100], "color": "rgba(34,197,94,0.15)"},
                    ],
                    "threshold": {
                        "line": {"color": "#94A3B8", "width": 2},
                        "thickness": 0.75,
                        "value": 75,
                    },
                },
            )
        )
        fig.update_layout(
            height=280,
            margin=dict(l=20, r=20, t=20, b=20),
            paper_bgcolor="rgba(15,23,42,0)",
            font_color="#E2E8F0",
        )
        return fig


Writing visualizer.py


In [5]:
%%writefile plant_config.py
"""
plant_config.py  v2.0
Topología actualizada según DIAGRAMA GENERACIÓN DE AGUAS — Coca-Cola Andina Chile
Unidades: m³/h

Cambios respecto a v1.0:
  1. Dos WUR separados: WUR_01 (rechazo NF) y WUR_02 (recuperación CIP)
  2. WUR_01 permeado → TK450 (LB03), WUR_02 permeado → TK300
  3. CIP recovery loop: LB02 → TK_CIP_REC → WUR_02
  4. Línea Benedictino completa: POZO_1B + PER → RO_PTR → TK_BENED → BENED_TOTAL
  5. POZO_1B (13.5 m³/h capacidad, Pozo 1 inferior) y PER como fuentes separadas
  6. ABAT flow calculado por solver (no fijado) para absorber ajuste de WUR_02
  7. Posiciones actualizadas según layout real del diagrama
"""
from __future__ import annotations
from models import PlantModel, UnitOperation, Stream


# Nivel de llenado operativo (~constante) de los estanques.
# Supuesto: los estanques se mantienen ~80% llenos y el nivel no cambia
# (o cambia marginalmente) → acumulación ≈ 0 → régimen estacionario.
# Sólo afecta el volumen operativo y el tiempo de residencia, NO el balance.
TANK_FILL_LEVEL = 0.80


def build_coca_cola_plant(empty: bool = False) -> PlantModel:
    """
    Planta Coca-Cola Chile — topología completa según diagrama oficial.

    RUTAS PRINCIPALES:
    ─────────────────
    [LB-02 Agua Proceso]
    POZOS → PRETRAT → TK300 → FFMM → NANO → FPUL → TK200 → LB02 → PROD/CIP

    [LB-03 Agua Servicio]
    FFMM → ABAT → TK450 → LB03 → SERV
    (WUR_01 permeado → TK450)

    [Recuperación rechazo NF]
    NANO → WUR_01 → TK450 (permeado) / DRAIN (rechazo)

    [Recuperación CIP]
    LB02 → TK_CIP_REC → WUR_02 → TK300 (permeado) / DRAIN (rechazo)

    [Línea Benedictino]
    POZO_1B + PER → PRETRAT_B → RO_PTR → TK_BENED → BENED_TOTAL
    (Agua para Benedictino FSR + LB-05/06 Lavado Envases)

    BALANCE ESPERADO (m³/h, día típico producción):
    ─────────────────────────────────────────────
    Entrada total:   134.3 (Pozos+Potable+Pozo1B+PER)
    LB02:            ~57.6
    LB03 servicio:   ~44.8 (Abat + WUR01 permeado)
    Benedictino:     ~10.8
    Útil total:      ~113.2
    Pérdidas:        ~21.1
    Eficiencia:      ~84%
    """
    model = PlantModel("Planta Tratamiento Agua — Coca-Cola Andina Chile")

    # ─────────────────────────────────────────
    # UNIDADES
    # (id, nombre, tipo, capacidad m³/h, x, y)
    # ─────────────────────────────────────────
    _units = [
        # ── Fuentes captación principal ──
        ("POZOS",      "Pozos 5,6,7+1A\n(Captación)",    "pozo",       480.0,  0.02, 0.82),
        ("POTABLE",    "Agua Potable\n(ESVAL)",           "pozo",       None,   0.02, 0.65),

        # ── Tratamiento principal ──
        ("PRETRAT",    "Pretrat.\n(AZUD+AG+)",            "filtro",     600.0,  0.15, 0.82),
        ("TK300",      "TK-300\nAgua Cruda",              "tanque",     480.0,  0.28, 0.74),
        ("FFMM",       "Filt. Multimedia\n+UV+Pulidores", "filtro",     340.0,  0.42, 0.74),
        ("NANO",       "Nano 1-2\n(NF)",                  "nano",       220.0,  0.55, 0.87),
        ("WUR_01",     "RO WUR 01\n(Rechazo NF)",         "uf",          18.8,  0.63, 0.66),
        ("FPUL",       "Filt. Pulidores\n1µm",            "filtro",     100.0,  0.67, 0.87),
        ("TK200",      "TK-200kk\n(200 m³)",              "tanque",     200.0,  0.76, 0.87),

        # ── Agua servicio (LB-03) ──
        ("ABAT",       "Abatidores\n(Vigaflow)",          "ablandador", None,   0.55, 0.58),
        ("TK450",      "TK-450\nAgua Blanda",             "tanque",     100.0,  0.68, 0.50),

        # ── Distribución ──
        ("LB02",       "LB-02\nAgua Proceso",             "mezclador",  None,   0.87, 0.87),
        ("LB03",       "LB-03\nAgua Servicio",            "mezclador",  None,   0.82, 0.46),

        # ── Recuperación CIP ──
        ("TK_CIP_REC", "TK Nano CIP\n(80 m³)",           "tanque",      80.0,  0.76, 0.64),
        ("WUR_02",     "RO WUR 02\n(CIP Recovery)",       "uf",         None,   0.83, 0.57),

        # ── Línea Benedictino ──
        ("POZO_1B",    "Pozo 1\nInferior (13.5)",         "pozo",        13.5,  0.02, 0.30),
        ("PER",        "Fuente PER",                      "pozo",       None,   0.02, 0.16),
        ("PRETRAT_B",  "Filtros + UV\nBenedictino",       "filtro",     None,   0.16, 0.24),
        ("RO_PTR",     "RO-PTR\n(rec=80%)",               "osmosis",    None,   0.36, 0.20),
        ("TK_BENED",   "TK-200\nBenedictino + Ozono",    "tanque",     200.0,  0.55, 0.20),

        # ── Sinks ──
        ("CIP_SINK",   "CIP +\nEnjuague",                 "cip",        None,   0.97, 0.97),
        ("PROD",       "Producción\nBebida",               "otro",       None,   0.97, 0.76),
        ("SERV",       "Servicios\n(ZH+Torres+Cond)",      "otro",       None,   0.97, 0.46),
        ("BENED_TOTAL","Benedictino\n(FSR + LB-05/06)",   "otro",       None,   0.82, 0.20),
        ("DRAIN",      "Drenaje\n(Pérdidas)",              "perdida",    None,   0.76, 0.10),
    ]

    for uid, name, op_type, cap, x, y in _units:
        model.add_unit(UnitOperation(
            id=uid, name=name, op_type=op_type,
            capacity=cap, x_pos=x, y_pos=y,
        ))

    # ─────────────────────────────────────────
    # CORRIENTES
    # (id, nombre, origen, destino, caudal|None, tipo)
    # None = incógnita → solver la calcula
    # ─────────────────────────────────────────
    _streams = [

        # ══ ENTRADAS PRINCIPALES ══════════════
        ("S01", "Agua cruda (Pozos 5,6,7+1A)",  "POZOS",    "PRETRAT",    116.5, "normal"),
        ("S02", "Agua potable (ESVAL)",          "POTABLE",  "TK300",        2.8, "normal"),

        # ══ PRETRATAMIENTO ═══════════════════
        ("S03", "Pretratado → TK-300",           "PRETRAT",  "TK300",       None, "normal"),    # UNKNOWN
        ("S04", "Rechazo AZUD + AG+",            "PRETRAT",  "DRAIN",        4.3, "rechazo"),

        # ══ RECIRCULACIONES A TK-300 ══════════
        ("S05", "Retro FFMM → TK-300",           "FFMM",     "TK300",        2.0, "recirculacion"),  # retrolavado FFMM recuperado
        ("S06", "WUR-02 permeado → TK-300",      "WUR_02",   "TK300",        1.7, "recirculacion"),  # CIP recovery → TK300

        # ══ TK-300 → FFMM ════════════════════
        ("S07", "TK-300 → FFMM",                 "TK300",    "FFMM",        None, "normal"),    # UNKNOWN (solver calcula todo el balance TK300)

        # ══ FFMM OUTPUTS ═════════════════════
        ("S08", "FFMM → Nanofiltración",         "FFMM",     "NANO",        74.4, "normal"),    # Total Nano METROSCUB
        ("S09", "FFMM → Abatidores",             "FFMM",     "ABAT",        None, "normal"),    # UNKNOWN (ajusta con WUR02)
        ("S10", "Retro FFMM → Drenaje",          "FFMM",     "DRAIN",        1.7, "perdida"),

        # ══ NANOFILTRACIÓN ════════════════════
        ("S11", "Permeado Nano → Pulidores",     "NANO",     "FPUL",        None, "normal"),    # UNKNOWN
        ("S12", "Rechazo Nano → WUR-01",         "NANO",     "WUR_01",      13.7, "rechazo"),   # WUR1 Entrada METROSCUB
        ("S13", "Rechazo Nano → Drenaje directo","NANO",     "DRAIN",        1.1, "rechazo"),   # diferencia total-WUR1

        # ══ WUR-01 (Rechazo NF Recovery) ══════
        ("S14", "WUR-01 permeado → TK-450",      "WUR_01",   "TK450",        6.2, "recirculacion"),  # WUR1 permeado → agua blanda
        ("S15", "WUR-01 rechazo → Drenaje",      "WUR_01",   "DRAIN",       None, "rechazo"),   # UNKNOWN (balance WUR01)

        # ══ FILTROS PULIDORES + TK-200kk ══════
        ("S16", "Pulidores → TK-200kk",          "FPUL",     "TK200",       None, "normal"),    # UNKNOWN
        ("S17", "Pérdida Pulidores",             "FPUL",     "DRAIN",        2.0, "perdida"),

        ("S18", "TK-200kk → LB-02",              "TK200",    "LB02",        None, "normal"),    # UNKNOWN

        # ══ ABATIDORES + TK-450 ═══════════════
        ("S19", "Abatidores → TK-450",           "ABAT",     "TK450",       None, "normal"),    # UNKNOWN
        ("S20", "Pérdida Abatidores",            "ABAT",     "DRAIN",        2.0, "perdida"),
        ("S21", "TK-450 → LB-03",               "TK450",    "LB03",        None, "normal"),    # UNKNOWN

        # ══ LB-02 DISTRIBUCIÓN ════════════════
        ("S22", "LB-02 → CIP + Enjuague",        "LB02",     "CIP_SINK",     6.3, "normal"),   # CIP directo drain = 9.6-3.3
        ("S23", "LB-02 → CIP Recovery",          "LB02",     "TK_CIP_REC",   3.3, "normal"),   # CIP a recuperación WUR02
        ("S24", "LB-02 → Producción",            "LB02",     "PROD",        None, "normal"),    # UNKNOWN

        # ══ LB-03 SERVICIOS ══════════════════
        ("S25", "LB-03 → Servicios",             "LB03",     "SERV",        None, "normal"),    # UNKNOWN

        # ══ RECUPERACIÓN CIP (WUR-02) ══════════
        ("S26", "TK CIP REC → WUR-02",           "TK_CIP_REC","WUR_02",     None, "normal"),    # UNKNOWN (balance TK_CIP_REC)
        ("S27", "WUR-02 rechazo → Drenaje",      "WUR_02",   "DRAIN",        1.6, "rechazo"),   # WUR2 rechazo METROSCUB
        # S06 (WUR_02→TK300) ya definido arriba

        # ══ LÍNEA BENEDICTINO ════════════════
        ("S28", "Pozo 1 Inferior",               "POZO_1B",  "PRETRAT_B",   10.0, "normal"),   # Estimado (cap=13.5)
        ("S29", "Fuente PER",                    "PER",      "PRETRAT_B",    5.0, "normal"),   # Estimado
        ("S30", "Pretrat.Bened → RO-PTR",        "PRETRAT_B","RO_PTR",      None, "normal"),    # UNKNOWN
        ("S31", "Pérdida Pretrat.Bened.",        "PRETRAT_B","DRAIN",        1.5, "perdida"),
        ("S32", "RO-PTR permeado → TK Bened.",   "RO_PTR",   "TK_BENED",    None, "normal"),    # UNKNOWN (recovery=0.80)
        ("S33", "RO-PTR rechazo → Drenaje",      "RO_PTR",   "DRAIN",       None, "rechazo"),   # UNKNOWN
        ("S34", "TK Bened. → Benedictino Total", "TK_BENED", "BENED_TOTAL", None, "normal"),    # UNKNOWN
    ]

    for sid, name, src, tgt, flow, stype in _streams:
        model.add_stream(Stream(
            id=sid, name=name,
            source_id=src, target_id=tgt,
            flow=flow, stream_type=stype,
        ))

    # ── Volumen de almacenamiento (m³) de los tanques, según diagrama ──
    # NOTA: `capacity` es CAUDAL nominal (m³/h). Para un tanque el dato
    #       físico relevante es el VOLUMEN (m³) → se guarda en `volume_m3`.
    #       Antes ambos se confundían en `capacity`; aquí quedan separados.
    #       No altera el balance estacionario (Σent = Σsal no depende del volumen).
    _TANK_VOLUMES_M3 = {
        "TK300":      480.0,   # Agua Cruda
        "TK200":      200.0,   # Agua Tratada (TK-200kk)
        "TK450":      100.0,   # Agua Blanda
        "TK_CIP_REC":  80.0,   # TK Nano CIP
        "TK_BENED":   200.0,   # Benedictino + Ozono
    }
    for _tid, _vol in _TANK_VOLUMES_M3.items():
        if _tid in model.units:
            model.units[_tid].volume_m3  = _vol
            model.units[_tid].fill_level = TANK_FILL_LEVEL
            model.units[_tid].capacity   = None   # un tanque no tiene caudal nominal
            # Nivel ~constante → acumulación ≈ 0 → balance estacionario
            # (accumulation se deja en None).

    # ── Restricción de recuperación RO-PTR (80%) ──
    model.units["RO_PTR"].recovery = 0.80

    # Arranque en blanco: sin caudales precargados (todas las corrientes
    # quedan como incógnita). Los valores de referencia del modelo siguen
    # definidos arriba y se usan con empty=False.
    if empty:
        for _s in model.streams.values():
            _s.flow = None

    return model


# ─────────────────────────────────────────────────────────────────
# Valores de referencia (METROSCUB día típico martes mayo 2026)
# ─────────────────────────────────────────────────────────────────
STREAM_REFERENCE = {
    "S01": {"fuente": "METROSCUB R9 /24",      "valor_ref": 116.5, "desc": "Consumo pozos principal"},
    "S02": {"fuente": "METROSCUB R11 /24",     "valor_ref":   2.8, "desc": "Agua Potable ESVAL"},
    "S04": {"fuente": "Estimado retro AZUD",   "valor_ref":   4.3, "desc": "Rechazo pretrat."},
    "S05": {"fuente": "METROSCUB R17 /24",     "valor_ref":   2.0, "desc": "Retro FFMM recuperado"},
    "S06": {"fuente": "METROSCUB R32 /24",     "valor_ref":   1.7, "desc": "WUR2 permeado"},
    "S08": {"fuente": "METROSCUB R20 /24",     "valor_ref":  74.4, "desc": "Total Nano feed"},
    "S10": {"fuente": "METROSCUB R14-R17 /24", "valor_ref":   1.7, "desc": "Retro FFMM drain"},
    "S12": {"fuente": "METROSCUB R26 /24",     "valor_ref":  13.7, "desc": "WUR1 Entrada (rechazo NF→WUR01)"},
    "S13": {"fuente": "METROSCUB R23-R26 /24", "valor_ref":   1.1, "desc": "Rechazo nano directo drain"},
    "S14": {"fuente": "METROSCUB R27 /24",     "valor_ref":   6.2, "desc": "WUR1 permeado→TK450"},
    "S17": {"fuente": "Estimado operacional",  "valor_ref":   2.0, "desc": "Pérd. pulidores"},
    "S20": {"fuente": "Estimado operacional",  "valor_ref":   2.0, "desc": "Pérd. abatidores"},
    "S22": {"fuente": "METROSCUB R36+R37-CIP_REC /24", "valor_ref": 6.3, "desc": "CIP directo drain"},
    "S23": {"fuente": "WUR2_perm+WUR2_rech",   "valor_ref":   3.3, "desc": "CIP a recuperación"},
    "S27": {"fuente": "METROSCUB R34 /24",     "valor_ref":   1.6, "desc": "WUR2 rechazo"},
    "S28": {"fuente": "Diagrama: cap 13.5 m³/h", "valor_ref": 10.0, "desc": "Pozo 1 inferior Benedictino"},
    "S29": {"fuente": "Estimado",              "valor_ref":   5.0, "desc": "Fuente PER"},
}


# ─────────────────────────────────────────────────────────────────
# Inventario de VOLÚMENES de estanques (m³) según el diagrama.
# Incluye estanques de subsistemas aún NO modelados como nodo (⚠).
# Es dato de referencia: no interviene en el balance estacionario.
# ─────────────────────────────────────────────────────────────────
DIAGRAM_TANK_VOLUMES_M3 = {
    # ── En el modelo (topología actual) ──
    "TK300 — Agua Cruda":                  480.0,
    "TK200 — Agua Tratada":                200.0,
    "TK450 — Agua Blanda":                 100.0,
    "TK_CIP_REC — TK Nano CIP":             80.0,
    "TK_BENED — Benedictino + Ozono":      200.0,
    # ── En el diagrama pero SIN nodo aún (revisar lectura) ──
    "Agua Filtrada":                       250.0,   # ⚠ tren principal
    "Agua Producto (línea bebida)":         20.0,   # ⚠ post-desareadores
    "Agua Producto (manantial/remin.)":    165.0,   # ⚠ aprox., línea SO-30
    "Estanque Acumulación Benedictino":     80.0,   # ⚠ línea inferior
    "Estanque Recuperación":                20.0,   # ⚠ aprox.
}


# ─────────────────────────────────────────────────────────────────
# Membranas: recuperación LIMPIA (post-CIP) y ensuciamiento por ciclo de CIP.
# Calibrado con METROSCUB enero–mayo 2026 (días representativos, P90/P10 reciente):
#   Recup. limpia (P90 Abr-May): Nano 89%, WUR-1 56%, WUR-2 58%, RO 80%.
#   φ_máx por ciclo (1 − P10/P90): Nano 17%, WUR-1 18%, WUR-2 49%.
# OJO: la Nano además se DEGRADA mes a mes (Ene 91% → May 80%, ≈ −2.5 pts/mes):
#   ensuciamiento IRREVERSIBLE (envejecimiento), que el CIP no recupera.
# cip_interval_days = intervalo de CIP ASUMIDO (editable en la app).
# ─────────────────────────────────────────────────────────────────
CLEAN_RECOVERY = {
    "NANO":   0.89,
    "WUR_01": 0.56,
    "WUR_02": 0.58,
    "RO_PTR": 0.80,
}

MEMBRANE_DECAY = {
    "NANO":   {"phi_max": 0.17, "cip_interval_days": 30, "model": "lineal"},
    "WUR_01": {"phi_max": 0.18, "cip_interval_days": 30, "model": "lineal"},
    "WUR_02": {"phi_max": 0.49, "cip_interval_days": 30, "model": "lineal"},
    "RO_PTR": {"phi_max": 0.10, "cip_interval_days": 30, "model": "lineal"},
}


Writing plant_config.py


In [6]:
%%writefile data_loader.py
"""
data_loader.py  v2.0
Carga datos diarios desde la hoja METROSCUB del archivo Excel del mapa de agua.
Mapeos actualizados para planta v2.0 (WUR_01, WUR_02, línea Benedictino).

Uso:
    from data_loader import load_daily_flows, available_dates
    fechas = available_dates('/path/to/05_Mapa_Agua_Mayo.xlsx')
    flows  = load_daily_flows('/path/to/file.xlsx', date(2026,5,5))
    # flows = {'S01': 116.5, 'S02': 2.8, 'S08': 74.4, ...}
"""
from __future__ import annotations
import datetime
import tempfile
import os
from typing import Dict, List, Optional, Tuple
import openpyxl


# ─────────────────────────────────────────────────────────────
# Mapeo METROSCUB (col B) → etiqueta auxiliar para cálculo
# Nota: las etiquetas auxiliares (_xxx) son intermedias;
#       la función _raw_to_flows las combina en stream IDs.
# ─────────────────────────────────────────────────────────────
_ROW_TO_STREAM: dict[str, dict] = {
    # ── ENTRADAS ──────────────────────────────────────────────
    "Consumo pozos":             {"ids": ["_pozos_m3dia"]},
    "Agua Potable (25%)":        {"ids": ["_potable_m3dia"]},

    # ── PRETRATAMIENTO ────────────────────────────────────────
    "Retroavado F. Azud":        {"ids": ["_retro_azud"]},         # rechazo AZUD+AG

    # ── TK-300 recirculaciones ────────────────────────────────
    "Tanque Recuperacion TK300": {"ids": ["_retro_ffmm_rec"]},     # retro FFMM recuperado → S05
    "Retrolavado FFMM":          {"ids": ["_retro_ffmm_total"]},   # total retro FFMM (para calcular drain S10)

    # ── NANOFILTRACIÓN ────────────────────────────────────────
    "Total Nano":                {"ids": ["_total_nano"]},         # feed NF → S08
    "Rechazo nano":              {"ids": ["_rechazo_nano"]},       # rechazo NF total

    # ── WUR-01 (Rechazo NF recovery) ─────────────────────────
    "WUR 1 Entrada":             {"ids": ["_wur1_entrada"]},       # NF reject → WUR_01 → S12
    "WUR 1 Permeado":            {"ids": ["_wur1_perm"]},          # WUR_01 perm → TK450 → S14
    "WUR 1 Rechazo":             {"ids": ["_wur1_rech"]},          # WUR_01 rech → DRAIN (info)

    # ── WUR-02 (CIP recovery) ─────────────────────────────────
    "WUR 2 Entrada T810":        {"ids": ["_wur2_entrada"]},       # WUR_02 feed (info)
    "WUR 2 Permeado":            {"ids": ["_wur2_perm"]},          # WUR_02 perm → TK300 → S06
    "WUR 2 Rechazo":             {"ids": ["_wur2_rech"]},          # WUR_02 rech → DRAIN → S27

    # ── CIP / LB-02 ───────────────────────────────────────────
    "CIP":                       {"ids": ["_cip"]},
    "ENJUAGUE DE LINEAS":        {"ids": ["_enjuague"]},

    # ── SERVICIOS LB-03 (referencia) ──────────────────────────
    "Agua Servicio":             {"ids": ["_agua_serv"]},          # solo lectura / referencia
    "Zona humeda":               {"ids": ["_zona_hum"]},
    "Torres Enfriamiento":       {"ids": ["_torres"]},
    "Condensadores Evaporativos":{"ids": ["_cond"]},
}


def available_dates(xlsx_path: str) -> List[datetime.date]:
    """Retorna lista de fechas disponibles en METROSCUB (fila 3)."""
    wb = openpyxl.load_workbook(xlsx_path, data_only=True, read_only=True)
    ws = wb["METROSCUB"]
    dates: List[datetime.date] = []
    for col in range(3, 45):
        val = ws.cell(row=3, column=col).value
        if isinstance(val, datetime.datetime):
            dates.append(val.date())
        elif val is None:
            break
    wb.close()
    return dates


def _read_raw_for_columns(ws, cols: List[int]) -> Dict[str, float]:
    """
    Lee las etiquetas mapeadas y PROMEDIA los valores diarios
    sobre las columnas indicadas (m³/día promedio del período).
    Ignora celdas vacías o cero.
    """
    raw: Dict[str, float] = {}
    for row in range(4, 120):
        label = ws.cell(row=row, column=2).value
        if label is None:
            continue
        label = str(label).strip()
        if label in _ROW_TO_STREAM:
            vals = []
            for col in cols:
                v = ws.cell(row=row, column=col).value
                if isinstance(v, (int, float)) and v > 0:
                    vals.append(float(v))
            if vals:
                raw[label] = sum(vals) / len(vals)
    return raw


def _raw_to_flows(raw: Dict[str, float]) -> Dict[str, float]:
    """
    Convierte etiquetas brutas (m³/día) a caudales por corriente (m³/h).
    Mapeo para plant_config.py v2.0

    Stream IDs relevantes:
      S01 POZOS_MAIN → PRETRAT             = Consumo pozos / 24
      S02 POTABLE → TK300                  = Agua Potable / 24
      S04 PRETRAT → DRAIN                  = Retroavado F. Azud × 2 / 24
      S05 FFMM → TK300 (retro rec.)        = Tanque Recuperacion TK300 / 24
      S06 WUR_02 → TK300                   = WUR 2 Permeado / 24
      S08 FFMM → NANO                      = Total Nano / 24
      S10 FFMM → DRAIN (retro drain)       = (Retrolavado FFMM − TK300 rec.) / 24
      S12 NANO → WUR_01 (rec. NF)          = WUR 1 Entrada / 24
      S13 NANO → DRAIN directo             = (Rechazo nano − WUR 1 Entrada) / 24
      S14 WUR_01 → TK450                   = WUR 1 Permeado / 24
      S22 LB02 → CIP_SINK (drain directo)  = (CIP+Enj) − (WUR2_perm+WUR2_rech) / 24
      S23 LB02 → TK_CIP_REC               = (WUR2 Permeado + WUR2 Rechazo) / 24
      S27 WUR_02 → DRAIN                   = WUR 2 Rechazo / 24
    """

    def m3h(m3dia: float) -> float:
        return round(m3dia / 24.0, 3)

    flows: Dict[str, float] = {}

    # ── Entradas ────────────────────────────────────────────
    if "Consumo pozos" in raw:
        flows["S01"] = m3h(raw["Consumo pozos"])

    if "Agua Potable (25%)" in raw:
        flows["S02"] = m3h(raw["Agua Potable (25%)"])

    # ── Pretratamiento ──────────────────────────────────────
    if "Retroavado F. Azud" in raw:
        flows["S04"] = m3h(raw["Retroavado F. Azud"] * 2)    # 2 filtros AZUD

    # ── TK-300 recirculaciones ──────────────────────────────
    if "Tanque Recuperacion TK300" in raw:
        flows["S05"] = m3h(raw["Tanque Recuperacion TK300"])  # retro FFMM recuperado

    # ── WUR-02 permeado → TK300 ─────────────────────────────
    if "WUR 2 Permeado" in raw:
        flows["S06"] = m3h(raw["WUR 2 Permeado"])

    # ── Nanofiltración feed ──────────────────────────────────
    if "Total Nano" in raw:
        flows["S08"] = m3h(raw["Total Nano"])

    # ── FFMM retro → DRAIN  (total - recuperado) ────────────
    if "Retrolavado FFMM" in raw and "Tanque Recuperacion TK300" in raw:
        retro_drain = raw["Retrolavado FFMM"] - raw["Tanque Recuperacion TK300"]
        if retro_drain > 0:
            flows["S10"] = m3h(retro_drain)

    # ── NF rechazo: split WUR-01 vs drain directo ───────────
    wur1_ent = raw.get("WUR 1 Entrada", 0)
    if wur1_ent > 0:
        flows["S12"] = m3h(wur1_ent)                          # NF reject → WUR_01
    rec_nano_total = raw.get("Rechazo nano", 0)
    if rec_nano_total > 0 and wur1_ent > 0:
        directo = max(0, rec_nano_total - wur1_ent)
        if directo > 0:
            flows["S13"] = m3h(directo)                       # NF reject → drain directo

    # ── WUR-01 permeado → TK450 ─────────────────────────────
    if "WUR 1 Permeado" in raw:
        flows["S14"] = m3h(raw["WUR 1 Permeado"])

    # ── WUR-02 rechazo → DRAIN ──────────────────────────────
    if "WUR 2 Rechazo" in raw:
        flows["S27"] = m3h(raw["WUR 2 Rechazo"])

    # ── CIP: split directo-drain vs CIP-recovery ────────────
    cip_total = raw.get("CIP", 0) + raw.get("ENJUAGUE DE LINEAS", 0)
    wur2_perm = raw.get("WUR 2 Permeado", 0)
    wur2_rech = raw.get("WUR 2 Rechazo", 0)
    cip_recovery = wur2_perm + wur2_rech   # agua CIP que pasó por WUR_02
    if cip_total > 0:
        cip_directo = max(0, cip_total - cip_recovery)
        flows["S22"] = m3h(cip_directo)                       # LB02 → CIP_SINK
    if cip_recovery > 0:
        flows["S23"] = m3h(cip_recovery)                      # LB02 → TK_CIP_REC

    return flows


# ─────────────────────────────────────────────────────────────
# API pública
# ─────────────────────────────────────────────────────────────

def load_daily_flows(
    xlsx_path: str,
    target_date: datetime.date,
) -> Dict[str, float]:
    """Caudales (m³/h) para un único día `target_date`."""
    wb = openpyxl.load_workbook(xlsx_path, data_only=True)
    ws = wb["METROSCUB"]

    date_col: Optional[int] = None
    for col in range(3, 50):
        val = ws.cell(row=3, column=col).value
        if isinstance(val, datetime.datetime) and val.date() == target_date:
            date_col = col
            break

    if date_col is None:
        wb.close()
        raise ValueError(
            f"Fecha {target_date} no encontrada en METROSCUB. "
            "Usa available_dates() para ver fechas disponibles."
        )

    raw = _read_raw_for_columns(ws, [date_col])
    wb.close()
    return _raw_to_flows(raw)


def load_range_flows(
    xlsx_path: str,
    start_date: datetime.date,
    end_date: datetime.date,
) -> Tuple[Dict[str, float], int]:
    """
    Caudales (m³/h) PROMEDIO entre start_date y end_date (inclusive).
    Retorna (flows, n_dias).
    """
    if end_date < start_date:
        start_date, end_date = end_date, start_date

    wb = openpyxl.load_workbook(xlsx_path, data_only=True)
    ws = wb["METROSCUB"]

    cols: List[int] = []
    for col in range(3, 50):
        val = ws.cell(row=3, column=col).value
        if isinstance(val, datetime.datetime):
            if start_date <= val.date() <= end_date:
                cols.append(col)

    if not cols:
        wb.close()
        raise ValueError(
            f"No hay fechas entre {start_date} y {end_date} en METROSCUB."
        )

    raw = _read_raw_for_columns(ws, cols)
    wb.close()
    return _raw_to_flows(raw), len(cols)


def load_multi_flows(
    paths: List[str],
) -> Tuple[Dict[str, float], int, int]:
    """Caudales (m³/h) promediados sobre TODOS los días representativos de
    varios archivos (p. ej. marzo + abril juntos). Promedio combinado,
    ponderado por día. Retorna (flows, n_dias_total, n_archivos)."""
    raw_vals: Dict[str, List[float]] = {}
    total_days = 0
    files_ok = 0
    for path in paths:
        try:
            wb = openpyxl.load_workbook(path, data_only=True, read_only=True)
            ws = wb["METROSCUB"]
        except Exception:
            continue
        cols = []
        for col in range(3, 50):
            v = ws.cell(row=3, column=col).value
            if isinstance(v, datetime.datetime):
                cols.append(col)
        if not cols:
            wb.close()
            continue
        files_ok += 1
        day_cols = set()
        for row in range(4, 120):
            label = ws.cell(row=row, column=2).value
            if label is None:
                continue
            label = str(label).strip()
            if label in _ROW_TO_STREAM:
                for col in cols:
                    v = ws.cell(row=row, column=col).value
                    if isinstance(v, (int, float)) and v > 0:
                        raw_vals.setdefault(label, []).append(float(v))
                        day_cols.add(col)
        total_days += len(day_cols)
        wb.close()
    raw_avg = {k: sum(vs) / len(vs) for k, vs in raw_vals.items() if vs}
    return _raw_to_flows(raw_avg), total_days, files_ok


def daily_membrane_recovery(
    paths: List[str],
) -> Dict[str, List[Tuple[datetime.date, float]]]:
    """Recuperación DIARIA (no promediada) de cada equipo de membrana sobre uno
    o varios archivos. Retorna {equipo: [(fecha, recuperacion_frac), ...]}.
    Recuperación = permeado / alimentación, con los valores de CADA día."""

    def _find(ws, *contains, exclude=()):
        for r in range(1, 160):
            v = ws.cell(row=r, column=2).value
            if v is None:
                continue
            s = str(v).strip().lower()
            if all(k in s for k in contains) and not any(x in s for x in exclude):
                return r
        return None

    series: Dict[str, List[Tuple[datetime.date, float]]] = {
        "Nano": [], "WUR-1": [], "WUR-2": [],
    }
    for path in paths:
        try:
            wb = openpyxl.load_workbook(path, data_only=True, read_only=True)
            ws = wb["METROSCUB"]
        except Exception:
            continue
        date_cols = []
        for col in range(3, 60):
            v = ws.cell(row=3, column=col).value
            if isinstance(v, datetime.datetime):
                date_cols.append((col, v.date()))
        r_nf = _find(ws, "total", "nano")
        r_nr = _find(ws, "rechazo", "nano", exclude=("agua", "%", "dif"))
        r_w1f = _find(ws, "wur 1", "entrada")
        r_w1p = _find(ws, "wur 1", "permeado")
        r_w2t = _find(ws, "wur 2", "entrada", "t810") or _find(ws, "wur 2", "entrada", exclude=("nano",))
        r_w2n = _find(ws, "wur 2", "entrada", "nano")
        r_w2p = _find(ws, "wur 2", "permeado")

        def val(r, col):
            if not r:
                return None
            x = ws.cell(row=r, column=col).value
            return float(x) if isinstance(x, (int, float)) else None

        for col, d in date_cols:
            nf, nr = val(r_nf, col), val(r_nr, col)
            if nf and nf > 1000 and nr is not None:
                rec = (nf - nr) / nf
                if 0.2 <= rec <= 1.0:
                    series["Nano"].append((d, rec))
            w1f, w1p = val(r_w1f, col), val(r_w1p, col)
            if w1f and w1f > 100 and w1p is not None:
                rec = w1p / w1f
                if 0.2 <= rec <= 1.0:
                    series["WUR-1"].append((d, rec))
            w2f = (val(r_w2t, col) or 0) + (val(r_w2n, col) or 0)
            w2p = val(r_w2p, col)
            if w2f > 100 and w2p is not None:
                rec = w2p / w2f
                if 0.2 <= rec <= 1.0:
                    series["WUR-2"].append((d, rec))
        wb.close()
    for k in series:
        series[k].sort(key=lambda t: t[0])
    return {k: v for k, v in series.items() if v}


def apply_flows_to_model(
    model,
    flows: Dict[str, float],
    overwrite_known: bool = True,
) -> int:
    """Aplica el dict de flows al PlantModel. Retorna # corrientes actualizadas."""
    updated = 0
    for sid, flow in flows.items():
        if sid not in model.streams:
            continue
        if not overwrite_known and model.streams[sid].flow is not None:
            continue
        model.set_flow(sid, flow if flow > 0 else None)
        updated += 1
    return updated


Writing data_loader.py


In [7]:
%%writefile requirements.txt
streamlit>=1.32.0
pandas>=2.0.0
numpy>=1.24.0
networkx>=3.1
plotly>=5.18.0
openpyxl>=3.1.0


Writing requirements.txt


In [8]:
%%writefile app.py
"""
app.py
Balance de Masa — Planta de Tratamiento de Agua
Interfaz Streamlit con 6 pestañas.

Ejecutar:
    cd water_balance
    streamlit run app.py
"""
from __future__ import annotations
import sys
import os
import io
import datetime
import pandas as pd
import numpy as np
import streamlit as st
import plotly.graph_objects as go

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

from models import (
    PlantModel, Stream, UnitOperation, STREAM_TYPES, UNIT_TYPES,
    accumulation_from_volume, accumulation_from_level, accumulation_from_flows,
)
from solver import MassBalanceSolver, fouling_from_time, fouling_from_cip
from visualizer import SankeyVisualizer
from plant_config import build_coca_cola_plant, CLEAN_RECOVERY, MEMBRANE_DECAY
from data_loader import (load_daily_flows, load_range_flows, load_multi_flows,
                         daily_membrane_recovery, available_dates, apply_flows_to_model)


# ─────────────────────────────────────────
# Configuración de página
# ─────────────────────────────────────────

st.set_page_config(
    page_title="Balance de Masa — Agua Industrial",
    page_icon="💧",
    layout="wide",
    initial_sidebar_state="expanded",
)

# CSS oscuro / industrial
st.markdown(
    """
    <style>
    /* fondo app */
    .main .block-container {padding-top: 1.2rem; padding-bottom: 1rem;}
    /* métricas */
    [data-testid="metric-container"] {
        background: rgba(30,41,59,0.7);
        border: 1px solid rgba(148,163,184,0.15);
        border-radius: 8px;
        padding: 14px 16px;
    }
    [data-testid="stMetricValue"] {font-size: 1.6rem !important;}
    /* tabs */
    .stTabs [data-baseweb="tab-list"] {gap: 4px;}
    .stTabs [data-baseweb="tab"] {
        background: rgba(30,41,59,0.5);
        border-radius: 6px;
        padding: 6px 14px;
        font-size: 0.85rem;
    }
    /* tabla */
    .stDataFrame {border-radius: 8px; overflow: hidden;}
    /* botones */
    div.stButton > button {
        background: linear-gradient(135deg, #0EA5E9, #0284C7);
        color: white; border: none; border-radius: 6px;
        font-weight: 600; padding: 8px 20px;
    }
    div.stButton > button:hover {background: linear-gradient(135deg, #38BDF8, #0EA5E9);}
    </style>
    """,
    unsafe_allow_html=True,
)


# ─────────────────────────────────────────
# Session state
# ─────────────────────────────────────────

def _init():
    if "model" not in st.session_state:
        st.session_state.model: PlantModel = build_coca_cola_plant(empty=True)
    if "solved" not in st.session_state:
        st.session_state.solved: dict = {}
    if "diag" not in st.session_state:
        st.session_state.diag: dict = {}
    if "kpis" not in st.session_state:
        st.session_state.kpis: dict = {}
    if "auto_solve" not in st.session_state:
        st.session_state.auto_solve: bool = False
    if "fouling" not in st.session_state:
        st.session_state.fouling: dict = {}
    if "daily_recovery" not in st.session_state:
        st.session_state.daily_recovery: dict = {}


_init()
model: PlantModel = st.session_state.model


def _solve_model(_m, _tol, fouling=None):
    """Resuelve el balance. Incorpora la acumulación (definida en los estanques)
    y, si hay factor de ensuciamiento (φ>0), la recuperación efectiva de las
    membranas (más rechazo). Devuelve (solved, kpis, diag)."""
    _solver = MassBalanceSolver(_m, tolerance=_tol)
    base_solved, base_diag = _solver.solve()
    foul = {u: v for u, v in (fouling or {}).items() if v and v > 0}
    if not foul:
        return base_solved, _solver.compute_kpis(base_solved), base_diag
    # recuperación limpia = la del punto de operación cargado (φ reduce desde ahí)
    be = _solver.compute_equipment_efficiency(base_solved)
    clean = {u: (be[u]["recuperacion_pct"] or 0) / 100.0 for u in be}
    solved, kpis, diag = _solver.simulate_fouling(foul, clean)
    diag = dict(diag)
    diag["messages"] = list(diag.get("messages", [])) + [
        "🧫 Ensuciamiento aplicado: φ = " + ", ".join(
            "{} {:.0f}%".format(u, v * 100) for u, v in foul.items())]
    return solved, kpis, diag


def _interpret_residual(equipo: str, residuo: float) -> str:
    """Interpretación cualitativa de un desbalance por equipo."""
    signo = "exceso de entrada" if residuo > 0 else "exceso de salida"
    eq = equipo.lower()
    if "wur" in eq:
        return f"{signo}. El WUR recibe alimentación extra (T810/nanos) no modelada"
    if "tk-300" in eq or "tk300" in eq:
        return f"{signo}. Posible recirculación o aporte no medido"
    if "pretrat" in eq or "multimedia" in eq:
        return f"{signo}. Propagado desde captación/retrolavados"
    if "nano" in eq:
        return f"{signo}. Revisar medición de permeado/rechazo"
    if abs(residuo) < 0.5:
        return "despreciable"
    return signo


# ─────────────────────────────────────────
# Sidebar
# ─────────────────────────────────────────

with st.sidebar:
    st.title("💧 Balance de Masa")
    st.caption("Planta Tratamiento de Agua — Coca-Cola Chile")
    st.divider()

    # ── Acciones rápidas ──
    st.subheader("Acciones")

    tolerancia = st.slider(
        "Tolerancia de balance (m³/h)",
        min_value=0.05, max_value=10.0, value=2.0, step=0.25,
        help="Desbalance máximo aceptable por equipo. Datos reales: 2-3 m³/h. Datos de diseño: 0.05",
    )

    col_a, col_b = st.columns(2)
    with col_a:
        if st.button("↺ Reset", use_container_width=True, help="Reiniciar planta con los caudales en blanco"):
            st.session_state.model = build_coca_cola_plant(empty=True)
            st.session_state.solved = {}
            st.session_state.diag = {}
            st.session_state.kpis = {}
            st.rerun()
    with col_b:
        if st.button("⚖️ Resolver", use_container_width=True, help="Ejecutar solver de balance de masa"):
            solved, kpis, diag = _solve_model(model, tolerancia, st.session_state.get("fouling"))
            st.session_state.solved = solved
            st.session_state.diag = diag
            st.session_state.kpis = kpis
            st.rerun()

    st.divider()

    # ── Cargar desde Excel ──
    st.subheader("📂 Cargar datos Excel")
    uploaded = st.file_uploader(
        "Archivo(s) Mapa Agua (.xlsx)",
        type=["xlsx"],
        label_visibility="collapsed",
        accept_multiple_files=True,
    )
    st.caption("Puedes subir **varios meses** a la vez (se promedian).")

    if uploaded:
        import tempfile
        paths = []
        for _i, _uf in enumerate(uploaded):
            _p = os.path.join(tempfile.gettempdir(), "_mapa_agua_%d.xlsx" % _i)
            with open(_p, "wb") as _f:
                _f.write(_uf.read())
            paths.append(_p)

        if len(paths) > 1:
            # ── Varios Excel: promedio combinado de todos sus días ──
            st.caption("📚 **%d archivos** → balance con el **promedio combinado** "
                       "de todos sus días." % len(paths))
            for _uf in uploaded:
                st.caption("• " + _uf.name)
            overwrite_m = st.checkbox("Sobreescribir valores existentes", value=True, key="ow_multi")
            if st.button("📥 Aplicar datos (promedio combinado)", use_container_width=True):
                try:
                    flows, n_dias, n_arch = load_multi_flows(paths)
                    n = apply_flows_to_model(
                        st.session_state.model, flows, overwrite_known=overwrite_m,
                    )
                    st.session_state.daily_recovery = daily_membrane_recovery(paths)
                    st.session_state.periodo = {"ini": None, "fin": None,
                                                "dias": n_dias, "archivos": n_arch}
                    st.success("%d corrientes • promedio de %d archivos (%d días)"
                               % (n, n_arch, n_dias))
                    st.session_state.solved = {}
                    st.rerun()
                except Exception as e:
                    st.error("Error combinando archivos: %s" % e)
        else:
            tmp_path = paths[0]
            try:
                fechas = available_dates(tmp_path)
                if fechas:
                    col_f1, col_f2 = st.columns(2)
                    with col_f1:
                        fecha_ini = st.selectbox(
                            "Desde",
                            options=fechas,
                            index=0,
                            format_func=lambda d: d.strftime("%d/%m/%Y"),
                        )
                    with col_f2:
                        fechas_fin = [d for d in fechas if d >= fecha_ini]
                        fecha_fin = st.selectbox(
                            "Hasta",
                            options=fechas_fin,
                            index=len(fechas_fin) - 1,
                            format_func=lambda d: d.strftime("%d/%m/%Y"),
                        )

                    n_dias_prev = len([d for d in fechas if fecha_ini <= d <= fecha_fin])
                    if fecha_ini == fecha_fin:
                        st.caption(f"📅 Análisis de **1 día**")
                    else:
                        st.caption(f"📅 Análisis de **{n_dias_prev} días** (promedio del período)")

                    overwrite = st.checkbox("Sobreescribir valores existentes", value=True)
                    if st.button("📥 Aplicar datos", use_container_width=True):
                        try:
                            if fecha_ini == fecha_fin:
                                flows = load_daily_flows(tmp_path, fecha_ini)
                                n_dias = 1
                            else:
                                flows, n_dias = load_range_flows(tmp_path, fecha_ini, fecha_fin)
                            n = apply_flows_to_model(
                                st.session_state.model, flows,
                                overwrite_known=overwrite,
                            )
                            st.session_state.daily_recovery = daily_membrane_recovery([tmp_path])
                            st.session_state.periodo = {
                                "ini": fecha_ini,
                                "fin": fecha_fin,
                                "dias": n_dias,
                            }
                            st.success(f"{n} corrientes • promedio de {n_dias} día(s)")
                            st.session_state.solved = {}
                            st.rerun()
                        except Exception as e:
                            st.error(f"Error: {e}")
                else:
                    st.warning("No se encontraron fechas en METROSCUB")
            except Exception as e:
                st.error(f"Error leyendo archivo: {e}")

    st.divider()

    # ── Info del modelo ──
    summ = model.summary()
    st.subheader("📊 Resumen modelo")
    st.metric("Nodos", summ["nodos"])
    c1, c2 = st.columns(2)
    c1.metric("Corrientes", summ["corrientes"])
    c2.metric("Incógnitas", summ["desconocidas"])

    if st.session_state.diag:
        status = st.session_state.diag.get("status", "")
        if status == "ok":
            st.success("Balance ✅ cerrado")
        elif status == "inconsistente":
            st.error("Balance ❌ inconsistente")
        elif status == "sub_especificado":
            st.warning("Balance ⚠️ sub-especificado")


# ─────────────────────────────────────────
# Tabs principales
# ─────────────────────────────────────────

tab1, tab2, tab3, tab4, tab5, tab6, tab7 = st.tabs([
    "🏭 Topología",
    "📊 Datos / Caudales",
    "⚖️ Balance",
    "🌊 Sankey",
    "🕸️ Diagrama",
    "📈 KPIs",
    "💧 Pérdidas",
])


# ─────────────────────────────────────────
# TAB 1 — Topología
# ─────────────────────────────────────────

with tab1:
    st.header("Topología del proceso")

    col_u, col_s = st.columns([1, 1.4])

    with col_u:
        st.subheader(f"Equipos ({summ['nodos']})")
        df_units = model.units_df()
        if not df_units.empty:
            st.dataframe(
                df_units[["ID", "Nombre", "Tipo", "Caudal nom. (m³/h)", "Volumen (m³)"]],
                hide_index=True,
                use_container_width=True,
            )

        # Agregar equipo
        with st.expander("➕ Agregar equipo"):
            with st.form("add_unit_form"):
                u_id    = st.text_input("ID único (ej: OSMOSIS_B)")
                u_name  = st.text_input("Nombre")
                u_type  = st.selectbox("Tipo", UNIT_TYPES)
                u_cap   = st.number_input("Caudal nominal (m³/h)", min_value=0.0, value=0.0)
                u_vol   = st.number_input("Volumen almacenamiento (m³)", min_value=0.0, value=0.0)
                c_x, c_y = st.columns(2)
                u_x = c_x.number_input("Posición X (0-1)", 0.0, 1.0, 0.5)
                u_y = c_y.number_input("Posición Y (0-1)", 0.0, 1.0, 0.5)
                submitted = st.form_submit_button("Agregar")
                if submitted and u_id and u_name:
                    if u_id in model.units:
                        st.error(f"ID '{u_id}' ya existe")
                    else:
                        model.add_unit(UnitOperation(
                            id=u_id, name=u_name, op_type=u_type,
                            capacity=u_cap if u_cap > 0 else None,
                            volume_m3=u_vol if u_vol > 0 else None,
                            x_pos=u_x, y_pos=u_y,
                        ))
                        st.success(f"Equipo '{u_name}' agregado")
                        st.rerun()

    with col_s:
        st.subheader(f"Corrientes ({summ['corrientes']})")
        df_str = model.streams_df()
        if not df_str.empty:
            display_df = df_str.copy()
            display_df["Estado"] = display_df["Caudal (m³/h)"].apply(
                lambda x: "✅ Conocida" if pd.notna(x) else "❓ Incógnita"
            )
            st.dataframe(
                display_df[["ID", "Nombre", "Desde", "Hacia", "Caudal (m³/h)", "Tipo", "Estado"]],
                hide_index=True,
                use_container_width=True,
            )

        # Agregar corriente
        unit_ids = list(model.units.keys())
        with st.expander("➕ Agregar corriente"):
            with st.form("add_stream_form"):
                s_id    = st.text_input("ID único (ej: S23)")
                s_name  = st.text_input("Nombre de la corriente")
                s_src   = st.selectbox("Origen", unit_ids, key="src_sel")
                s_tgt   = st.selectbox("Destino", unit_ids, key="tgt_sel")
                s_flow  = st.number_input("Caudal m³/h (0 = desconocido)", 0.0, 9999.0, 0.0)
                s_type  = st.selectbox("Tipo", STREAM_TYPES)
                submitted2 = st.form_submit_button("Agregar")
                if submitted2 and s_id and s_name and s_src != s_tgt:
                    model.add_stream(Stream(
                        id=s_id, name=s_name,
                        source_id=s_src, target_id=s_tgt,
                        flow=s_flow if s_flow > 0 else None,
                        stream_type=s_type,
                    ))
                    st.success(f"Corriente '{s_name}' agregada")
                    st.rerun()

    # ── Nivel de estanque (ajuste manual: % o metros) ──
    with st.expander("📏 Nivel de estanque (manual: volumen o altura)"):
        _tk_lvl = [(uid, u) for uid, u in model.units.items()
                   if u.op_type == "tanque" and u.volume_m3]
        if not _tk_lvl:
            st.warning("No hay estanques con volumen definido.")
        else:
            lvl_id = st.selectbox(
                "Estanque",
                options=[uid for uid, _ in _tk_lvl],
                format_func=lambda x: model.units[x].name.replace("\n", " "),
                key="lvl_tank",
            )
            u_lvl = model.units[lvl_id]
            modo_lvl = st.radio(
                "Ajustar por", ["Volumen (m³)", "Altura (m)"],
                horizontal=True, key="lvl_modo",
            )
            nuevo_fill = u_lvl.fill_level if u_lvl.fill_level is not None else 1.0
            nueva_alt = u_lvl.height_m
            if modo_lvl == "Volumen (m³)":
                v_act = st.number_input(
                    "Volumen actual (m³)", min_value=0.0,
                    max_value=float(u_lvl.volume_m3),
                    value=float(min(u_lvl.volume_m3, (u_lvl.fill_level or 1.0) * u_lvl.volume_m3)),
                    step=1.0, key="lvl_vol",
                )
                nuevo_fill = (v_act / u_lvl.volume_m3) if u_lvl.volume_m3 else 0.0
            else:
                c_h1, c_h2 = st.columns(2)
                nueva_alt = c_h1.number_input(
                    "Altura total (m)", min_value=0.1,
                    value=float(u_lvl.height_m) if u_lvl.height_m else 5.0,
                    step=0.1, key="lvl_H",
                )
                h_act = c_h2.number_input(
                    "Nivel actual (m)", min_value=0.0, max_value=nueva_alt,
                    value=float(min(nueva_alt, (u_lvl.fill_level or 1.0) * (u_lvl.height_m or nueva_alt))),
                    step=0.1, key="lvl_h",
                )
                nuevo_fill = (h_act / nueva_alt) if nueva_alt > 0 else 0.0
            v_op = u_lvl.volume_m3 * nuevo_fill
            st.caption(
                f"→ Llenado **{nuevo_fill*100:.0f}%** · Volumen operativo "
                f"**{v_op:.1f} m³** de {u_lvl.volume_m3:g} m³"
                + (f" · Nivel **{nuevo_fill*nueva_alt:.2f} m** / {nueva_alt:g} m" if nueva_alt else "")
            )
            if st.button("💾 Aplicar nivel", use_container_width=True, key="lvl_btn"):
                model.units[lvl_id].fill_level = round(float(nuevo_fill), 4)
                if nueva_alt:
                    model.units[lvl_id].height_m = round(float(nueva_alt), 3)
                st.session_state.solved = {}
                st.success(f"{u_lvl.name.replace(chr(10), ' ')} → nivel {nuevo_fill*100:.0f}%")
                st.rerun()

with tab2:
    st.header("Ingreso de caudales medidos")
    st.caption(
        "Edita el campo **Caudal (m³/h)** directamente. "
        "Deja en blanco (borrar el valor) para marcar una corriente como **incógnita**."
    )

    df_edit = model.streams_df()[
        ["ID", "Nombre", "Desde", "Hacia", "Caudal (m³/h)", "Tipo", "Temp (°C)", "Conductividad"]
    ].copy()

    # Nullable float para que el editor acepte celdas vacías
    df_edit["Caudal (m³/h)"] = pd.array(
        df_edit["Caudal (m³/h)"].tolist(), dtype=pd.Float64Dtype()
    )

    edited_df = st.data_editor(
        df_edit,
        column_config={
            "ID":            st.column_config.TextColumn("ID",       disabled=True, width="small"),
            "Nombre":        st.column_config.TextColumn("Nombre",   disabled=True, width="medium"),
            "Desde":         st.column_config.TextColumn("Desde",    disabled=True, width="small"),
            "Hacia":         st.column_config.TextColumn("Hacia",    disabled=True, width="small"),
            "Caudal (m³/h)": st.column_config.NumberColumn(
                "Caudal (m³/h)",
                format="%.3f",
                help="Valor medido. Vacío = incógnita (el solver la calcula)",
            ),
            "Tipo":          st.column_config.SelectboxColumn("Tipo", options=STREAM_TYPES, width="small"),
            "Temp (°C)":     st.column_config.NumberColumn("Temp °C",   format="%.1f"),
            "Conductividad": st.column_config.NumberColumn("Cond µS/cm", format="%.1f"),
        },
        use_container_width=True,
        hide_index=True,
        key="stream_data_editor",
        num_rows="fixed",
    )

    if st.button("💾 1) Aplicar cambios de caudal"):
        for _, row in edited_df.iterrows():
            sid = row["ID"]
            if sid not in model.streams:
                continue
            raw_flow = row["Caudal (m³/h)"]
            model.set_flow(sid, None if pd.isna(raw_flow) else float(raw_flow))
            s = model.streams[sid]
            temp = row.get("Temp (°C)"); cond = row.get("Conductividad")
            s.temperature  = float(temp) if pd.notna(temp) else None
            s.conductivity = float(cond) if pd.notna(cond) else None
        st.session_state.solved = {}
        st.success("Caudales actualizados.")

    st.divider()

    # ── 2) Acumulación en estanques ──
    with st.expander("♒ 2) Acumulación en estanques (opcional)"):
        st.caption("Caudal por estanque (m³/h):  **+** se llena · **−** se vacía · **0** nivel estable.")
        _tanks = [(uid, u) for uid, u in model.units.items() if u.op_type == "tanque"]
        _accols = st.columns(3)
        _acc_in = {}
        for _i, (_uid, _u) in enumerate(_tanks):
            _cur = _u.accumulation if _u.accumulation is not None else 0.0
            _acc_in[_uid] = _accols[_i % 3].number_input(
                _u.name.replace("\n", " ")[:18], value=float(_cur), step=0.5,
                format="%.2f", key="accf_" + _uid)
        if st.button("Aplicar acumulación", key="apply_acc_flow"):
            for _uid, _v in _acc_in.items():
                model.units[_uid].accumulation = _v if abs(_v) > 1e-9 else None
            st.session_state.solved = {}
            st.success("Acumulación aplicada.")

    st.divider()

    # ── 3) Resolver el balance (con acumulación) ──
    if st.button("⚖️ 3) Resolver balance de masa", use_container_width=True, type="primary"):
        for _, row in edited_df.iterrows():
            sid = row["ID"]
            if sid not in model.streams:
                continue
            raw_flow = row["Caudal (m³/h)"]
            model.set_flow(sid, None if pd.isna(raw_flow) else float(raw_flow))
        solved, kpis, diag = _solve_model(model, tolerancia, st.session_state.get("fouling"))
        st.session_state.solved = solved
        st.session_state.diag  = diag
        st.session_state.kpis  = kpis
        st.rerun()

    # ── Alerta: faltan datos para resolver (corrientes sin determinar) ──
    _diag = st.session_state.get("diag", {})
    if _diag.get("status") == "sub_especificado":
        _falt = _diag.get("faltantes", [])
        st.error(
            "⚠️ **Faltan datos para resolver el balance de masa.**  \n"
            "Ingresa el caudal de estas corrientes en la tabla de arriba "
            "(basta con algunas) y vuelve a Resolver:"
        )
        if _falt:
            st.markdown("\n".join("- **%s**" % f for f in _falt))


# ─────────────────────────────────────────
# TAB 3 — Balance
# ─────────────────────────────────────────

with tab3:
    st.header("Resultados del balance de masa")

    if not st.session_state.solved:
        st.info("⚠️ Pulsa **⚖️ Resolver** en la barra lateral (o en la pestaña Datos) para ejecutar el solver.")
    else:
        diag   = st.session_state.diag
        solved = st.session_state.solved

        # ── Estado con bandas de aceptación ──
        residuals = diag.get("residuals", {})
        max_res = max((abs(v) for v in residuals.values()), default=0.0)
        status = diag.get("status", "")

        if status == "ok" and not residuals:
            st.success("✅ Balance cerrado correctamente")
        elif status == "sub_especificado":
            _falt = diag.get("faltantes", [])
            if _falt:
                st.warning("⚠️ Sistema sub-especificado — faltan datos.\n\n"
                           "**Corrientes sin determinar** (ingresa alguna como dato): "
                           + ", ".join(_falt))
            else:
                st.warning("⚠️ Sistema sub-especificado — faltan datos")
        elif status == "sin_ecuaciones":
            st.warning("⚠️ Sin ecuaciones — agregar nodos con entradas y salidas")
        elif max_res > 0:
            # Comparar contra el caudal de entrada para dar % de error
            entrada = solved.get("S01", 0) + solved.get("S02", 0)
            pct = (max_res / entrada * 100) if entrada > 0 else 0
            if pct <= 8:
                st.warning(
                    f"⚠️ Balance aceptable con desviaciones de medición. "
                    f"Desbalance máx: {max_res:.2f} m³/h ({pct:.1f}% de la entrada). "
                    f"Normal en datos reales de planta."
                )
            else:
                st.error(
                    f"❌ Balance con desviación alta: {max_res:.2f} m³/h ({pct:.1f}% de la entrada). "
                    f"Revisar caudalímetros o corrientes no medidas."
                )

        # ── Residuos por nodo ──
        if residuals:
            with st.expander(f"🔍 Diagnóstico — dónde no reconcilian los datos ({len(residuals)} equipos)", expanded=True):
                st.caption(
                    "Cada residuo = (entradas − salidas) del equipo. "
                    "Un residuo alto indica una corriente no medida o un caudalímetro descalibrado."
                )
                res_df = pd.DataFrame(
                    [{"Equipo": k, "Desbalance (m³/h)": round(v, 3),
                      "Interpretación": _interpret_residual(k, v)}
                     for k, v in sorted(residuals.items(), key=lambda x: -abs(x[1]))]
                )
                st.dataframe(
                    res_df, hide_index=True, use_container_width=True,
                    column_config={
                        "Desbalance (m³/h)": st.column_config.NumberColumn(format="%.3f"),
                    },
                )

        st.divider()

        # ── Tabla de resultados ──
        st.subheader("Caudales calculados")
        rows = []
        for sid, s in model.streams.items():
            src = model.units.get(s.source_id)
            tgt = model.units.get(s.target_id)
            flow_solved = solved.get(sid, 0.0)
            flow_orig   = s.flow
            was_unknown = flow_orig is None
            rows.append({
                "ID":              sid,
                "Corriente":       s.name,
                "Desde":           src.name if src else "",
                "Hacia":           tgt.name if tgt else "",
                "Caudal medido":   flow_orig,
                "Caudal resuelto": round(flow_solved, 3),
                "Tipo":            s.stream_type,
                "Calculada":       "✅ Sí" if was_unknown else "— No",
            })

        res_df = pd.DataFrame(rows).sort_values("Caudal resuelto", ascending=False)
        st.dataframe(
            res_df,
            hide_index=True,
            use_container_width=True,
            column_config={
                "Caudal medido":   st.column_config.NumberColumn(format="%.3f"),
                "Caudal resuelto": st.column_config.NumberColumn(format="%.3f"),
            },
        )

        # ── Descargar resultados ──
        csv = res_df.to_csv(index=False).encode("utf-8")
        st.download_button(
            "📥 Descargar CSV",
            data=csv,
            file_name="balance_resultado.csv",
            mime="text/csv",
        )


# ─────────────────────────────────────────
# TAB 4 — Sankey
# ─────────────────────────────────────────

with tab4:
    st.header("Diagrama Sankey")

    if not st.session_state.solved:
        st.info("Ejecuta primero el solver (sidebar → ⚖️ Resolver)")

        # Mostrar Sankey con valores actuales (pueden tener incógnitas = 0)
        cur_flows = {sid: (s.flow or 0.0) for sid, s in model.streams.items()}
        fig_s = SankeyVisualizer.create_sankey(model, cur_flows)
        st.plotly_chart(fig_s, use_container_width=True)
        st.caption("ℹ️ Las corrientes desconocidas aparecen con caudal 0 hasta resolver.")
    else:
        col_s1, col_s2 = st.columns([4, 1])
        with col_s2:
            min_flow = st.number_input(
                "Caudal mínimo a mostrar (m³/h)",
                min_value=0.0, max_value=50.0, value=0.1, step=0.05,
            )
        fig_s = SankeyVisualizer.create_sankey(
            model, st.session_state.solved, min_flow=min_flow
        )
        with col_s1:
            st.plotly_chart(fig_s, use_container_width=True)

    # Leyenda de colores
    st.caption(
        "**Colores de corrientes:** "
        "🔵 Normal &nbsp;|&nbsp; 🟠 Recirculación &nbsp;|&nbsp; "
        "🔴 Rechazo &nbsp;|&nbsp; ⚫ Pérdida &nbsp;|&nbsp; 🟢 Producto"
    )


# ─────────────────────────────────────────
# TAB 5 — Diagrama de Proceso
# ─────────────────────────────────────────

with tab5:
    st.header("Diagrama de proceso")

    flows_for_graph = st.session_state.solved or {
        sid: (s.flow or 0.0) for sid, s in model.streams.items()
    }

    fig_g = SankeyVisualizer.create_process_graph(model, flows_for_graph)
    st.plotly_chart(fig_g, use_container_width=True)

    if not st.session_state.solved:
        st.caption("⚠️ Las corrientes desconocidas se muestran como 0 hasta resolver el balance.")


# ─────────────────────────────────────────
# TAB 6 — KPIs
# ─────────────────────────────────────────

with tab6:
    st.header("KPIs — Eficiencia hídrica")

    # Mostrar período analizado si existe
    periodo = st.session_state.get("periodo")
    if periodo:
        if periodo.get("archivos"):
            st.caption(
                f"📅 Promedio combinado de **{periodo['archivos']} archivos** "
                f"({periodo['dias']} días)"
            )
        elif periodo["dias"] == 1:
            st.caption(
                f"📅 Período: **{periodo['ini'].strftime('%d/%m/%Y')}** (1 día)"
            )
        else:
            st.caption(
                f"📅 Período: **{periodo['ini'].strftime('%d/%m/%Y')} → "
                f"{periodo['fin'].strftime('%d/%m/%Y')}** "
                f"(promedio de {periodo['dias']} días)"
            )

    if not st.session_state.kpis:
        st.info("Ejecuta el solver primero para ver los KPIs.")
    else:
        kpis = st.session_state.kpis

        # ── Métricas principales ──
        col1, col2, col3, col4, col5 = st.columns(5)
        col1.metric(
            "Consumo Total",
            f"{kpis.get('consumo_pozos', 0):.1f} m³/h",
            help="Todos los pozos + potable + POZO_1B + PER",
        )
        col2.metric(
            "LB-02 Proceso",
            f"{kpis.get('agua_proceso_lb02', 0):.1f} m³/h",
            help="Agua distribuida al loop de proceso (bebida)",
        )
        col3.metric(
            "LB-03 Servicio",
            f"{kpis.get('agua_servicio_lb03', 0):.1f} m³/h",
            help="Agua distribuida al loop de servicio (ZH+Torres+Cond)",
        )
        col4.metric(
            "Benedictino",
            f"{kpis.get('agua_benedictino', 0):.1f} m³/h",
            help="Agua tratada para Benedictino FSR + LB-05/06",
        )
        col5.metric(
            "Eficiencia",
            f"{kpis.get('eficiencia_pct', 0):.1f}%",
            delta=f"{kpis.get('eficiencia_pct', 0) - 80:.1f}% vs obj. 80%",
            help="(LB02+LB03+Benedictino) / Consumo Total Pozos",
        )

        st.divider()

        col6, col7, col8 = st.columns(3)
        col6.metric(
            "Drenaje total",
            f"{kpis.get('rechazo_total', 0):.1f} m³/h",
            help="Suma de todas las corrientes hacia DRAIN",
        )
        col7.metric(
            "Pérdidas totales",
            f"{kpis.get('perdidas_totales', 0):.1f} m³/h",
            help="Pozos - (LB02+LB03+Benedictino)",
        )
        col8.metric(
            "Agua útil total",
            f"{kpis.get('agua_util_total', 0):.1f} m³/h",
            help="LB02 + LB03 + Benedictino",
        )

        # ── Gauge ──
        col_g, col_pie = st.columns([1, 1.5])

        with col_g:
            fig_gauge = SankeyVisualizer.create_efficiency_gauge(
                kpis.get("eficiencia_pct", 0)
            )
            st.plotly_chart(fig_gauge, use_container_width=True)

        with col_pie:
            st.subheader("Distribución de agua")
            labels = ["LB-02 Proceso", "LB-03 Servicio", "Benedictino",
                      "CIP+Enjuague", "Drenaje/Rechazo"]
            lb02   = kpis.get("agua_proceso_lb02", 0)
            lb03   = kpis.get("agua_servicio_lb03", 0)
            bened  = kpis.get("agua_benedictino", 0)
            pozos  = kpis.get("consumo_pozos", 1)
            drain  = kpis.get("rechazo_total", 0)
            cip    = max(0, pozos - lb02 - lb03 - bened - drain)
            perds  = kpis.get("perdidas_totales", 0)
            values = [lb02, lb03, bened, cip, drain]
            fig_pie = go.Figure(
                go.Pie(
                    labels=labels,
                    values=values,
                    hole=0.45,
                    marker=dict(
                        colors=["#0EA5E9", "#22C55E", "#8B5CF6", "#F59E0B", "#EF4444"],
                        line=dict(color="rgba(15,23,42,0.7)", width=2),
                    ),
                    textinfo="label+percent",
                    textfont=dict(size=11, color="#E2E8F0"),
                    hovertemplate="%{label}: %{value:.2f} m³/h (%{percent})<extra></extra>",
                )
            )
            fig_pie.update_layout(
                height=290,
                margin=dict(l=10, r=10, t=20, b=10),
                paper_bgcolor="rgba(15,23,42,0)",
                font_color="#E2E8F0",
                legend=dict(font=dict(size=10)),
                showlegend=True,
            )
            st.plotly_chart(fig_pie, use_container_width=True)

        # ── Tabla detalle por equipo ──
        st.divider()
        st.subheader("Balance por equipo")
        solved = st.session_state.solved
        rows_eq = []
        for uid, u in model.units.items():
            in_f = sum(
                solved.get(s.id, s.flow or 0) for s in model.inlets_of(uid)
            )
            out_f = sum(
                solved.get(s.id, s.flow or 0) for s in model.outlets_of(uid)
            )
            balance = round(in_f - out_f, 4)
            rows_eq.append({
                "Equipo":          u.name,
                "Tipo":            u.op_type,
                "Entrada (m³/h)":  round(in_f, 3),
                "Salida (m³/h)":   round(out_f, 3),
                "Δ Balance":       balance,
            })

        eq_df = pd.DataFrame(rows_eq).sort_values("Entrada (m³/h)", ascending=False)
        st.dataframe(
            eq_df,
            hide_index=True,
            use_container_width=True,
            column_config={
                "Entrada (m³/h)":  st.column_config.NumberColumn(format="%.3f"),
                "Salida (m³/h)":   st.column_config.NumberColumn(format="%.3f"),
                "Δ Balance":       st.column_config.NumberColumn(
                    format="%.4f",
                    help="< ±0.05 = balance cerrado",
                ),
            },
        )

        # ── Tiempo de residencia (nivel ~constante) ──
        st.divider()
        st.subheader("⏱️ Tiempo de residencia en estanques")
        st.caption(
            "τ = Volumen operativo / Caudal. Supone nivel ~constante "
            "(acumulación ≈ 0); volumen operativo = volumen × nivel de llenado."
        )
        rows_tau = []
        for uid, u in model.units.items():
            if u.op_type != "tanque" or u.volume_m3 is None:
                continue
            q = sum(solved.get(s.id, s.flow or 0) for s in model.inlets_of(uid))
            tau = u.residence_time_h(q)
            rows_tau.append({
                "Estanque":            u.name.replace("\n", " "),
                "Volumen (m³)":        u.volume_m3,
                "Nivel (%)":           round((u.fill_level or 1.0) * 100),
                "Vol. operativo (m³)": round(u.operating_volume_m3, 1) if u.operating_volume_m3 is not None else None,
                "Caudal (m³/h)":       round(q, 2),
                "τ (h)":               round(tau, 2) if tau is not None else None,
            })
        if rows_tau:
            st.dataframe(
                pd.DataFrame(rows_tau).sort_values("τ (h)", ascending=False),
                hide_index=True,
                use_container_width=True,
                column_config={
                    "τ (h)": st.column_config.NumberColumn(
                        format="%.2f h",
                        help="Tiempo medio que el agua permanece en el estanque",
                    ),
                },
            )
        else:
            st.info("No hay estanques con volumen definido.")

        # ── Eficiencia de equipos de tratamiento (membranas) ──
        st.divider()
        st.subheader("⚙️ Eficiencia de equipos (recuperación de membranas)")
        st.caption(
            "Recuperación = permeado / alimentación. Aplica a nanofiltración, "
            "ósmosis (RO) y unidades de recuperación WUR/UF."
        )
        eff = MassBalanceSolver(model).compute_equipment_efficiency(solved)
        if eff:
            rows_eff = [{
                "Equipo":              d["nombre"],
                "Tipo":                d["tipo"],
                "Alimentación (m³/h)": d["feed"],
                "Permeado (m³/h)":     d["permeado"],
                "Rechazo (m³/h)":      d["rechazo"],
                "Recuperación (%)":    d["recuperacion_pct"],
                "Diseño (%)":          d["recuperacion_diseno_pct"],
            } for d in eff.values()]
            st.dataframe(
                pd.DataFrame(rows_eff).sort_values("Recuperación (%)", ascending=False),
                hide_index=True,
                use_container_width=True,
                column_config={
                    "Recuperación (%)": st.column_config.NumberColumn(
                        format="%.1f %%", help="Permeado / alimentación"),
                    "Diseño (%)": st.column_config.NumberColumn(
                        format="%.1f %%", help="Recuperación de diseño, si está definida"),
                },
            )
            st.caption(
                "ℹ️ Recuperación alta = más agua aprovechada y menos rechazo. "
                "Si difiere mucho del diseño, revisar membranas u operación."
            )
        else:
            st.info("No hay equipos de membrana (nano/RO/UF) en el modelo.")

        # ── Ensuciamiento de membranas (fouling) ──
        st.divider()
        st.subheader("🧫 Ensuciamiento de membranas (fouling)")
        st.caption(
            "Al ensuciarse la membrana baja la recuperación efectiva "
            "(R_ef = R_limpia × (1 − φ)) → menos permeado y más rechazo a drenaje."
        )
        modo_foul = st.radio("Definir por", ["Eficiencia manual (%)", "Por ciclo de CIP (días)"],
                             horizontal=True, key="foul_modo")
        _memb = [u for u in ["NANO", "WUR_01", "WUR_02", "RO_PTR"] if u in model.units]
        phi = {}
        if modo_foul == "Eficiencia manual (%)":
            st.caption("100% = membrana limpia (recuperación completa) · 0% = totalmente tapada.")
            _cf = st.columns(len(_memb))
            for _c, _u in zip(_cf, _memb):
                _efm = _c.slider(model.units[_u].name.replace("\n", " ")[:16],
                                 0, 100, 100, key="foul_" + _u,
                                 help="100% = limpia · 0% = totalmente tapada")
                phi[_u] = 1.0 - _efm / 100.0
        else:
            _dias = st.number_input("Días desde el último CIP", 0, 365, 15, step=1, key="foul_dias")
            st.caption("Se asume un intervalo de CIP por membrana (editable). φ crece de 0 "
                       "(recién lavada) a φ_máx al cumplir el intervalo. φ_máx calibrado ene–may 2026.")
            _ci = st.columns(len(_memb))
            for _c, _u in zip(_ci, _memb):
                _d = MEMBRANE_DECAY.get(_u, {"phi_max": 0.2, "cip_interval_days": 30, "model": "lineal"})
                _intv = _c.number_input("CIP {} c/(días)".format(_u), 1, 365,
                                        int(_d["cip_interval_days"]), key="cip_" + _u)
                phi[_u] = fouling_from_cip(_dias, _intv, _d["phi_max"], _d["model"])
            st.caption(" · ".join("{}: φ={:.0f}%".format(_u, phi[_u] * 100) for _u in _memb))

        _sf = MassBalanceSolver(model, tolerance=tolerancia)
        base_s, base_k, _bd = _sf.simulate_fouling({}, CLEAN_RECOVERY)
        foul_s, foul_k, foul_d = _sf.simulate_fouling(phi, CLEAN_RECOVERY)
        _be = _sf.compute_equipment_efficiency(base_s)
        _fe = _sf.compute_equipment_efficiency(foul_s)
        rows_f = []
        for _u in _memb:
            b = _be.get(_u); f = _fe.get(_u)
            if not b or not f:
                continue
            rows_f.append({
                "Membrana":          b["nombre"],
                "φ (%)":             round(phi[_u] * 100),
                "Recup. limpia (%)": b["recuperacion_pct"],
                "Recup. efect. (%)": f["recuperacion_pct"],
                "Permeado (m³/h)":   f["permeado"],
                "Δ Permeado":        round(f["permeado"] - b["permeado"], 2),
                "Rechazo (m³/h)":    f["rechazo"],
                "Δ Rechazo":         round(f["rechazo"] - b["rechazo"], 2),
            })
        if rows_f:
            st.dataframe(pd.DataFrame(rows_f), hide_index=True, use_container_width=True)
        _cf1, _cf2, _cf3 = st.columns(3)
        _cf1.metric("Rechazo total a drenaje", "{:.1f} m³/h".format(foul_k["rechazo_total"]),
                    "{:+.1f}".format(foul_k["rechazo_total"] - base_k["rechazo_total"]),
                    delta_color="inverse")
        _cf2.metric("Eficiencia hídrica", "{:.1f} %".format(foul_k["eficiencia_pct"]),
                    "{:+.1f}".format(foul_k["eficiencia_pct"] - base_k["eficiencia_pct"]))
        _cf3.metric("Agua útil total", "{:.1f} m³/h".format(foul_k["agua_util_total"]),
                    "{:+.1f}".format(foul_k["agua_util_total"] - base_k["agua_util_total"]))
        st.caption("Comparación vs escenario limpio (φ=0). Calibración: METROSCUB mayo 2026.")


# ─────────────────────────────────────────
# TAB 7 — Pérdidas
# ─────────────────────────────────────────

with tab7:
    st.header("💧 Análisis automático de pérdidas")

    if not st.session_state.solved:
        st.info("Ejecuta el solver primero (sidebar → ⚖️ Resolver) para detectar las pérdidas.")
    else:
        solver_l = MassBalanceSolver(st.session_state.model)
        losses = solver_l.analyze_losses(st.session_state.solved)
        ranking = losses["ranking"]

        if not ranking:
            st.success("No se detectaron corrientes de pérdida en el modelo actual.")
        else:
            # ── Métricas resumen ──
            cL1, cL2, cL3 = st.columns(3)
            cL1.metric("Pérdida total", f"{losses['total_perdidas']:.1f} m³/h")
            cL2.metric("Consumo pozos", f"{losses['consumo_pozos']:.1f} m³/h")
            cL3.metric(
                "Pérdida global",
                f"{losses['pct_perdida_global']:.1f}%",
                help="% del agua de pozo que se pierde",
            )

            st.divider()

            # ── Ranking ──
            st.subheader("Ranking de pérdidas (mayor a menor)")
            rank_rows = []
            for i, l in enumerate(ranking, 1):
                rank_rows.append({
                    "#":               i,
                    "Corriente":       l["corriente"],
                    "Destino":         l["destino"],
                    "Categoría":       l["categoria"],
                    "Caudal (m³/h)":   l["caudal"],
                    "% Agua Pozo":     l["pct_pozos"],
                    "% Pérdidas":      l["pct_perdidas"],
                })
            rank_df = pd.DataFrame(rank_rows)
            st.dataframe(
                rank_df,
                hide_index=True,
                use_container_width=True,
                column_config={
                    "#":              st.column_config.NumberColumn(width="small"),
                    "Caudal (m³/h)":  st.column_config.NumberColumn(format="%.2f"),
                    "% Agua Pozo":    st.column_config.ProgressColumn(
                        format="%.1f%%", min_value=0,
                        max_value=max((l["pct_pozos"] for l in ranking), default=1),
                    ),
                    "% Pérdidas":     st.column_config.NumberColumn(format="%.1f%%"),
                },
            )

            # ── Resumen textual estilo informe ──
            st.caption("**Resumen:**  " + "  •  ".join(
                f"{i}. {l['corriente']} → {l['caudal']:.1f} m³/h → {l['pct_pozos']:.0f}% del agua de pozo"
                for i, l in enumerate(ranking[:3], 1)
            ))

            st.divider()

            # ── Gráficos ──
            cG1, cG2 = st.columns([1.3, 1])

            with cG1:
                st.subheader("Pérdidas por corriente")
                fig_bar = go.Figure(go.Bar(
                    x=[l["caudal"] for l in ranking],
                    y=[l["corriente"] for l in ranking],
                    orientation="h",
                    marker=dict(color="#EF4444"),
                    text=[f"{l['caudal']:.1f} m³/h ({l['pct_pozos']:.0f}%)" for l in ranking],
                    textposition="auto",
                    hovertemplate="%{y}<br>%{x:.2f} m³/h<extra></extra>",
                ))
                fig_bar.update_layout(
                    height=max(260, 40 * len(ranking)),
                    margin=dict(l=10, r=10, t=10, b=10),
                    paper_bgcolor="rgba(15,23,42,0)",
                    plot_bgcolor="rgba(22,33,55,0.5)",
                    font_color="#E2E8F0",
                    xaxis=dict(title="m³/h", gridcolor="rgba(148,163,184,0.15)"),
                    yaxis=dict(autorange="reversed"),
                )
                st.plotly_chart(fig_bar, use_container_width=True)

            with cG2:
                st.subheader("Por categoría")
                cats = losses["por_categoria"]
                fig_cat = go.Figure(go.Pie(
                    labels=list(cats.keys()),
                    values=list(cats.values()),
                    hole=0.45,
                    marker=dict(
                        colors=["#EF4444", "#F59E0B", "#8B5CF6", "#EC4899", "#64748B", "#0EA5E9"],
                        line=dict(color="rgba(15,23,42,0.7)", width=2),
                    ),
                    textinfo="label+percent",
                    textfont=dict(size=10, color="#E2E8F0"),
                    hovertemplate="%{label}: %{value:.2f} m³/h<extra></extra>",
                ))
                fig_cat.update_layout(
                    height=290,
                    margin=dict(l=10, r=10, t=10, b=10),
                    paper_bgcolor="rgba(15,23,42,0)",
                    font_color="#E2E8F0",
                    showlegend=False,
                )
                st.plotly_chart(fig_cat, use_container_width=True)

            # ── Descargar ──
            csv_l = rank_df.to_csv(index=False).encode("utf-8")
            st.download_button(
                "📥 Descargar ranking CSV",
                data=csv_l,
                file_name="ranking_perdidas.csv",
                mime="text/csv",
            )

    # ── Eficiencia de equipos de recuperación (membranas) en el tiempo ──
    st.divider()
    st.subheader("📈 Eficiencia de equipos de recuperación — por día")
    st.caption(
        "Recuperación diaria (permeado / alimentación) de cada equipo de membrana, "
        "calculada desde el Excel. Un punto por día, sin promediar. Un gráfico por equipo."
    )
    _dr = st.session_state.get("daily_recovery", {})
    if not _dr:
        st.info("Carga uno o más Excel (pestaña Datos) para ver la evolución de la eficiencia.")
    else:
        _col_eq = {"Nano": "#00838F", "WUR-1": "#1565C0", "WUR-2": "#6A1B9A", "RO-PTR": "#2E7D32"}
        for _eq, _serie in _dr.items():
            if not _serie:
                continue
            _fx = [p[0] for p in _serie]
            _fy = [round(p[1] * 100, 1) for p in _serie]
            _fig_eq = go.Figure(go.Scatter(
                x=_fx, y=_fy, mode="lines+markers", name=_eq,
                line=dict(color=_col_eq.get(_eq, "#0EA5E9")), marker=dict(size=6),
            ))
            _fig_eq.update_layout(
                title="%s — Recuperación diaria (%d días, prom %.0f%%)"
                      % (_eq, len(_serie), sum(_fy) / len(_fy)),
                xaxis_title="Fecha", yaxis_title="Recuperación (%)",
                height=300, margin=dict(l=10, r=10, t=40, b=10),
                yaxis=dict(rangemode="tozero"), showlegend=False,
            )
            st.plotly_chart(_fig_eq, use_container_width=True)


# ─────────────────────────────────────────
# Footer
# ─────────────────────────────────────────

st.divider()
st.caption(
    "Balance de Masa Hídrico v1.1 — "
    "Coca-Cola Chile | Ingeniería Química — Operaciones | "
    "Datos: METROSCUB (m³/h = m³/día ÷ 24)"
)


Writing app.py


## 3. Lanzar la aplicaciónEspera ~10 segundos tras ejecutar y haz clic en la URL **trycloudflare.com**.

In [ ]:
import subprocess, time, os, urllib.request

os.chdir('/content')

# Verificar que los archivos existen
faltan = [f for f in ['app.py','models.py','solver.py','visualizer.py','plant_config.py','data_loader.py'] if not os.path.exists(f)]
if faltan:
    print('❌ Faltan archivos:', faltan)
    print('   → Ejecuta primero las celdas de la seccion 2 (%%writefile)')
else:
    print('✅ Todos los archivos presentes\n')
    # Descargar cloudflared (tunel gratuito, sin cuenta)
    if not os.path.exists('cloudflared'):
        print('Descargando cloudflared...')
        urllib.request.urlretrieve(
            'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
            'cloudflared')
        os.chmod('cloudflared', 0o755)
    # Lanzar Streamlit en segundo plano
    subprocess.Popen(
        ['streamlit', 'run', 'app.py',
         '--server.port', '8501',
         '--server.headless', 'true',
         '--server.enableCORS', 'false',
         '--server.enableXsrfProtection', 'false'],
        stdout=open('/content/streamlit.log','w'),
        stderr=subprocess.STDOUT)
    time.sleep(8)
    print('Streamlit iniciado. Abriendo tunel...\n')
    print('='*60)
    print('  Haz clic en la URL https://....trycloudflare.com de abajo')
    print('='*60 + '\n')
    !./cloudflared tunnel --url http://localhost:8501

✅ Todos los archivos presentes

Descargando cloudflared...
Streamlit iniciado. Abriendo tunel...

  Haz clic en la URL https://....trycloudflare.com de abajo

2026-06-02T12:31:24Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-06-02T12:31:24Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-06-02T12:31:29Z INF +--------------------------------------------------------------------------------------------+
2026-06-02T12:31:29Z INF |  Your quick Tun

---### Si el túnel fallaDetén la celda anterior y ejecuta esto en una celda nueva:```python!npm install -g localtunnelimport subprocess, time, osos.chdir('/content')subprocess.Popen(['streamlit','run','app.py','--server.port','8501','--server.headless','true'])time.sleep(8)!npx localtunnel --port 8501```La contraseña del túnel localtunnel es la IP que devuelve: `!curl https://loca.lt/mytunnelpassword`